# Baseline figures

Use the bao kernel in VS Code in your allocated compute session. Choose saved inputs below, then edit and run the plot cells you need. Leave an input as `None` to skip that section.


In [ ]:
from pathlib import Path
import csv
import json
import os
import sys
import itertools

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pzbao/run.py").is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from pzbao.figures import data as readers, style

BINS, TIERS, STATES, PARAMETERS = readers.BINS, readers.TIERS, readers.STATES, readers.PARAMETERS
LABELS, ROBUSTNESS = readers.LABELS, readers.ROBUSTNESS
figures = {}
figure_captions = {}
supporting_tables = {}


def show(fig, name, caption=""):
    # Size typography for the actual printed canvas, before display and export.
    from matplotlib.text import Text

    for ax in fig.axes:
        ax.xaxis.label.set_fontsize(9)
        ax.yaxis.label.set_fontsize(9)
        ax.title.set_fontsize(9)
        ax.tick_params(axis="both", labelsize=8)
        legend = ax.get_legend()
        if legend is not None:
            for text in legend.get_texts():
                text.set_fontsize(8)
    for text in fig.findobj(Text):
        if text.get_text() and text.get_gid() != "gain-label" and 0 < text.get_fontsize() < 8:
            text.set_fontsize(8)
    figures[name] = fig
    figure_captions[name] = caption
    display(fig)
    if caption:
        display(Markdown(caption))
    plt.close(fig)


## Inspect a pipeline run

Set BAO_RUN_ROOT before starting the kernel, or set run_path below. Partial runs remain readable. This section uses the same native readers as the stage diagnostics.


In [ ]:
from pzbao.run import load_run, route_name, routes_for
from pzbao.diagnostics.review import collect_route
from pzbao.figures.data import (
    load_all_fits,
    load_chain_samples,
    load_xi_multipoles_with_std,
    render_table,
)
from pzbao.figures.style import two_col
from IPython.display import HTML, Image
import html

run_path = os.environ.get("BAO_RUN_ROOT")  # Or an explicit existing run directory
if run_path:
    RUN_ROOT = Path(run_path).expanduser().resolve()
    run = load_run(RUN_ROOT)
    routes = routes_for(run["config"], run["metadata"]["workload"])
    REVIEW_ROOT = RUN_ROOT / "review"

    print("Saved report:", REVIEW_ROOT / "report.html")

    def show_table(name, limit=20):
        path = REVIEW_ROOT / f"{name}.csv"
        if not path.exists():
            print(f"{name}: no saved table yet; regenerate the review when needed.")
            return []
        with path.open(newline="") as stream:
            rows = list(csv.DictReader(stream))
        if not rows:
            print(f"{name}: no available rows.")
            return rows
        fields = list(rows[0])
        heading = "".join(f"<th>{html.escape(field)}</th>" for field in fields)
        body = "".join(
            "<tr>" + "".join(f"<td>{html.escape(row[field])}</td>" for field in fields) + "</tr>"
            for row in rows[:limit]
        )
        display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))
        print(f"{name}: showing {min(limit, len(rows))} of {len(rows)} rows; full table: {path}")
        return rows

    coverage_rows = show_table("coverage")
    error_rows = show_table("errors")

    density_rows = show_table("density")
    boundary_rows = show_table("boundary_support")

    selected_route = routes[0]
    print("Selected route:", selected_route)
    route_record, _ = collect_route(run, selected_route)
    if route_record["coverage"].get("measurements") == "complete":
        multipoles = run["config"]["fitting"]["fit_surfaces"][selected_route["tracer"]][
            "observed_multipoles"
        ]
        separations, xi_mean, xi_scatter = load_xi_multipoles_with_std(
            RUN_ROOT,
            selectors=selected_route,
            multipoles=multipoles,
        )
        fig, ax = plt.subplots(figsize=two_col(3.5))
        for index, ell in enumerate(multipoles):
            ax.errorbar(
                separations,
                separations**2 * xi_mean[index],
                yerr=separations**2 * xi_scatter[index],
                label=rf"$\ell={ell}$",
            )
        ax.set(
            xlabel=r"$s\ [h^{-1}\,\mathrm{Mpc}]$",
            ylabel=r"$s^2\bar\xi_\ell\ [(h^{-1}\,\mathrm{Mpc})^2]$",
        )
        ax.legend()
        fig.tight_layout()
        show(fig, "native-correlation")
    else:
        print(
            "Complete measurement ensemble unavailable:",
            route_record["coverage"].get("measurements"),
        )

    profiles = load_all_fits(RUN_ROOT)
    individual_profiles = tuple(row for row in profiles if row["realization_id"] is not None)
    mean_profiles = tuple(row for row in profiles if row["realization_id"] is None)
    print(
        f"Available native profiles: {len(individual_profiles)} individual; {len(mean_profiles)} mean."
    )
    mean_profile_table = render_table(mean_profiles)
    print(mean_profile_table)
    paired_rows = show_table("paired_fits")
    mean_gain_rows = show_table("aggregate_gains")
    recovery_rows = show_table("profile_recovery")

    posterior_rows = show_table("posterior")
    sampler_rows = show_table("sampler")
    gr_rows = show_table("gr")
    chain_samples = None
    sampling_path = RUN_ROOT / "sample" / "units" / route_name(selected_route) / "sampling.json"
    if sampling_path.exists():
        parameter = "qiso"
        chain_samples = load_chain_samples(RUN_ROOT, selectors=selected_route, parameter=parameter)
        print("Posterior axes (ensemble, walker, retained iteration):", chain_samples.shape)
        print("Pooled posterior standard deviation:", float(chain_samples.std(ddof=1)))
        fig, ax = plt.subplots(figsize=two_col(3.5))
        for index, ensemble in enumerate(chain_samples):
            ax.hist(
                ensemble.ravel(), bins=40, density=True, histtype="step", label=f"Ensemble {index}"
            )
        ax.set(xlabel=parameter, ylabel="Posterior density")
        ax.legend()
        fig.tight_layout()
        show(fig, "native-posterior")
    else:
        print("No published complete ensemble collection for this route.")

    covariance_rows = show_table("covariance")
    damping_rows = show_table("damping")

    figure_paths = sorted(REVIEW_ROOT.glob("*.png"))
    print("\n".join(path.name for path in figure_paths) or "No saved figures yet.")
    # Set this to a filename listed above when ready to inspect a figure.
    FIGURE_NAME = None
    if FIGURE_NAME is not None:
        chosen = (REVIEW_ROOT / FIGURE_NAME).resolve()
        display(Image(filename=str(chosen)))


## Saved baseline

Choose the baseline report directory. The reader loads posterior samples, fit summaries and correlation arrays; the following cells control the plots.


In [ ]:
BASELINE_REPORT = None  # REPO / "products/reports/paper-baseline-refresh"
BASELINE_REPORT = globals().get("baseline_report_choice", BASELINE_REPORT)
# The per-bin baseline: the unified-prior refit of the per-bin primary, read from its run root
# together with the measurement run passed as SELECTION_RUN; the bundle is then not read.
FIT_RUN = globals().get("fit_run_choice")
SELECTION_RUN = globals().get("selection_run_choice")
PAPER_DIR = Path(globals().get("paper_dir_choice", REPO / "paper"))
DETECTION_BIN = "LRG1"
OVERLAY_BIN = "LRG1"
OVERLAY_CONDITION = "sigma_0.003"
if FIT_RUN is not None:
    if SELECTION_RUN is None:
        raise ValueError("the per-bin baseline needs its measurement run as SELECTION_RUN")
    if BASELINE_REPORT is not None:
        raise ValueError("choose the per-bin fit run or the saved bundle, not both")
    baseline = readers.load_perbin_baseline(FIT_RUN, SELECTION_RUN, detection_bin=DETECTION_BIN)
    BASELINE_REPORT = Path(FIT_RUN)  # the cells below are keyed on a chosen baseline
elif BASELINE_REPORT is not None:
    baseline = readers.load_baseline(BASELINE_REPORT, detection_bin=DETECTION_BIN)
if BASELINE_REPORT is not None:
    for relative in ("pzbao/figures/baseline.ipynb", "pzbao/figures/data.py",
                     "pzbao/figures/style.py", "pzbao/figures/render_baseline.py",
                     "pzbao/figures/forecasts.ipynb", "pzbao/catalog.py", "pzbao/theory.py",
                     "pzbao/fitting.py", "pzbao/sampling.py", "pzbao/artifacts.py",
                     "pzbao/storage.py", "pzbao/run.py", "pzbao/config/__init__.py",
                     "pzbao/config/schema_validation.py", "pzbao/config/stages.py",
                     "pzbao/clustering.py", "pzbao/covariance.py",
                     "pzbao/diagnostics/clustering.py", "pzbao/likelihood.py", "paper/prd.mplstyle",
                     "pzbao/config/sources/survey_pars_v28_base_cbe.yaml"):
        baseline["sources"].read_current(REPO / relative)
    for name in ("main.proposed.tex", "bao.bib"):
        baseline["sources"].read_current(PAPER_DIR / name)
    draws = baseline["draws"]
    fits = baseline["fits"]
    mean_fits = baseline["mean_fits"]
    profiles_full = baseline["profiles"]
    correlation_panels = baseline["correlation"]
    fom_pairs = baseline["fom"]
    joint_rows = baseline["joint"]
    supporting_tables.update({
        "posterior-summary": baseline["posterior_summary"],
        "posterior-records": baseline["posterior_records"],
        "profile-diagnostics": baseline["profile_diagnostics"],
        "fom-values": fom_pairs,
        "joint-constraint": joint_rows,
        "density": baseline["density"],
        "correlation": correlation_panels,
    })
    selection = None
    if SELECTION_RUN is not None:
        selection = readers.baseline_selection(
            SELECTION_RUN, baseline["measurement_identity"], baseline["sources"]
        )
        supporting_tables["selection-diagnostics"] = selection
        preparations, frozen_config, _ = readers.read_precision_sources(
            baseline["sources"], SELECTION_RUN, baseline["measurement_identity"]
        )
        dr1 = readers.dr1_reference(baseline["sources"].json(readers.DEFAULT_INPUT))
        precision_rows = readers.mock_rows(preparations, frozen_config) + readers.dr1_rows(dr1, frozen_config)
        z_eff_rows = readers.effective_redshift_rows(preparations, frozen_config)
        supporting_tables["effective-redshifts"] = z_eff_rows
        supporting_tables["precision-values"] = dict(
            rows=precision_rows, calibration=readers.calibration_summary(), dr1_reference=dr1
        )

SIGMA_LABEL_TO_VALUE = {
    "desi_baseline": 0.0,
    **{tier: float(tier.split("_")[1]) for tier in TIERS[1:]},
}


def posterior_draws(tracer, zmin, zmax, sigma_label, smoothing, recon, param):
    return draws[(tracer, zmin, zmax, sigma_label, smoothing, recon)][
        :, ("qiso", "qap").index(param)
    ]


In [ ]:
from pzbao.figures.style import (
    JOINT_COLOR,
    PRD_STYLE_FILE,
    SIGMA_BASELINE,
    SIGMA_BASELINE_COLOR,
    SIGMA_TIERS,
    ZBIN_GRID_PANELS,
    sigma_color,
    wide,
    zbin_grid_axes,
    zbin_panel_label,
    XI_ERRORBAR_KW,
    XI_MARKER_STRIDE,
    zbin_grid_axes_with_residual,
    two_col,
)
from scipy.ndimage import gaussian_filter
from matplotlib.patches import Patch


## One-dimensional posterior density

Fixed-bandwidth KDE, fixed seed subsampling, and the existing display window are preserved.


In [ ]:
import sys

from pathlib import Path

import matplotlib.pyplot as plt

import numpy as np

_QGRID = np.linspace(0.80, 1.20, 401)


def post1d__kde(samples: np.ndarray, bandwidth: float = 0.002) -> np.ndarray:
    """Cheap fixed-bandwidth Gaussian KDE on :data:`_QGRID`.

    Faster than scipy.stats.gaussian_kde for ~500k samples; deterministic
    bandwidth keeps cross-panel comparisons fair.
    """
    if len(samples) == 0:
        return np.zeros_like(_QGRID)
    # Subsample to 50k for speed without changing shape
    if len(samples) > 50_000:
        rng = np.random.default_rng(0)
        samples = rng.choice(samples, 50_000, replace=False)
    diff = (_QGRID[:, None] - samples[None, :]) / bandwidth
    kernel = np.exp(-0.5 * diff * diff) / (np.sqrt(2 * np.pi) * bandwidth)
    return kernel.mean(axis=1)


def joint_gaussian_density(grid: np.ndarray, mean: float, variance: float) -> np.ndarray:
    """Normal density of a joint-constraint marginal on *grid*; comparable to the tier KDEs."""
    if not np.isfinite(variance) or variance <= 0:
        raise ValueError("joint marginal variance must be positive")
    return np.exp(-0.5 * (grid - mean) ** 2 / variance) / np.sqrt(2 * np.pi * variance)


def joint_record_for(name: str, state: str = "postreconstruction"):
    """The available joint record of one bin and state, or None (no per-realization profiles)."""
    for record in joint_rows:
        route = record["route"]
        if route["measurement_bin"] == name and route["reconstruction_state"] == state:
            return record if record.get("status") == "available" else None
    return None


def post1d__populate_panel(ax, tracer: str, zmin: float, zmax: float, name: str = None) -> None:
    """One panel: α_iso KDE per photometric tier, postrecon, sm=15, plus the joint marginal."""
    for sigma_label, sigma_val in SIGMA_LABEL_TO_VALUE.items():
        if sigma_label == "desi_baseline":  # the unsmeared control is not a photometric tier
            continue
        samples = posterior_draws(
            tracer, zmin, zmax, sigma_label, smoothing=15, recon="postrecon", param="qiso"
        )
        if samples.size == 0:
            continue
        # Evaluate all retained draws; only the displayed axis is restricted.
        kde = post1d__kde(samples)
        ax.plot(_QGRID, kde, color=sigma_color(sigma_val), lw=1.0)
    joint = joint_record_for(name) if name is not None else None
    if joint is not None:
        index = joint["parameters"].index("qiso")
        ax.plot(_QGRID, joint_gaussian_density(_QGRID, joint["mean"][index],
                                               joint["covariance"][index][index]),
                color=JOINT_COLOR, lw=1.4, ls="--")
    ax.axvline(1.0, color="grey", lw=0.5, linestyle=":", zorder=1)
    ax.set_xlim(0.95, 1.05)
    # The joint Gaussian can exceed the tier densities: size the panel to every drawn curve.
    shown = (0.95 <= _QGRID) & (_QGRID <= 1.05)
    peak = max(
        (float(np.max(line.get_ydata()[shown])) for line in ax.lines
         if len(line.get_ydata()) == _QGRID.size),
        default=1.0,
    )
    ax.set_ylim(0, 1.08 * peak)
    ax.set_yticks([])  # arbitrary normalisation; suppress y ticks
    ax.grid(True, which="major", axis="x", alpha=0.3)


if BASELINE_REPORT is not None:
    plt.style.use(PRD_STYLE_FILE)
    fig, axes = zbin_grid_axes(
        figsize_kind="two_col",
        h=5.0,
        sharey=False,
        xlabel=r"$\alpha_{\rm iso}$",
        ylabel=r"$p(\alpha_{\rm iso})$",
    )
    for ax, name, (tracer, zmin, zmax, _label) in zip(axes[:5], BINS, ZBIN_GRID_PANELS):
        post1d__populate_panel(ax, tracer, zmin, zmax, name)
        zbin_panel_label(ax, tracer, zmin, zmax)
    legend_ax = axes[5]
    handles = []
    for sigma in SIGMA_TIERS:
        handles.append(plt.Line2D([0], [0], color=sigma_color(sigma), label=rf"$\sigma_0={sigma}$"))
    handles.append(plt.Line2D([0], [0], color=JOINT_COLOR, ls="--", lw=1.4,
                              label="joint (four classes), Gaussian"))
    legend_ax.legend(
        handles, [h.get_label() for h in handles], loc="center", frameon=False, fontsize="small"
    )
    show(fig, "fig_post1d")


## Joint posterior contours

The wide tiers remain included even when their contours lie outside the common window.


In [ ]:
import matplotlib.pyplot as plt

import numpy as np

from scipy.ndimage import gaussian_filter

# The four photometric tiers; the unsmeared control is not a sample. Broad 0.01 and 0.03
# contours are allowed to clip outside the common comparison window and stay in the legend.
_OVERLAY_SIGMAS = (0.001, 0.003, 0.01, 0.03)


def post2d__contour_levels(h: np.ndarray) -> tuple:
    """Return the (level_2σ, level_1σ) histogram values that enclose 86% / 39%."""
    flat = np.sort(h.ravel())[::-1]
    csum = np.cumsum(flat)
    csum /= csum[-1]
    lvl_1sigma = flat[np.searchsorted(csum, 0.393)]
    lvl_2sigma = flat[np.searchsorted(csum, 0.865)]
    return (lvl_2sigma, lvl_1sigma)


def post2d__collect_panel(tracer: str, zmin: float, zmax: float):
    """Load chains once for one z-bin: contour data + the σ≤0.003 zoom extent.

    Returns ``(contour_data, zoom)`` where ``contour_data`` is a list of
    ``(sigma_val, xc, yc, h, lvls)`` and ``zoom`` is the data-driven
    ``(xlo, xhi, ylo, yhi)`` view of the tight tiers (or None). Only the
    σ≤0.003 samples set the zoom — the broad, detached σ=0.01 and 0.03
    posteriors would otherwise stretch it.
    """
    contour_data = []
    all_qiso, all_qap = [], []
    for sigma_val in _OVERLAY_SIGMAS:
        sigma_label = next(
            (lbl for lbl, val in SIGMA_LABEL_TO_VALUE.items() if val == sigma_val), None
        )
        if sigma_label is None:
            continue
        qiso = posterior_draws(tracer, zmin, zmax, sigma_label, 15, "postrecon", "qiso")
        qap = posterior_draws(tracer, zmin, zmax, sigma_label, 15, "postrecon", "qap")
        if qiso.size == 0 or qap.size == 0:
            continue
        # Sub-sample for speed
        if qiso.size > 100_000:
            rng = np.random.default_rng(0)
            idx = rng.choice(qiso.size, 100_000, replace=False)
            qiso = qiso[idx]
            qap = qap[idx]
        # Histogram each tier over its OWN data-driven range — the old fixed
        # wide range left tight posteriors spanning only a few bins, which
        # made the contours blocky. Then Gaussian-smooth the histogram before
        # computing enclosed-mass levels (corner.py-style); the padded
        # percentile range covers essentially all posterior mass, so the
        # levels remain correct.
        xlo, xhi = np.percentile(qiso, [0.1, 99.9])
        ylo, yhi = np.percentile(qap, [0.1, 99.9])
        xpad = 0.3 * (xhi - xlo) + 1e-4
        ypad = 0.3 * (yhi - ylo) + 1e-4
        h, xe, ye = np.histogram2d(
            qiso,
            qap,
            bins=120,
            range=[[xlo - xpad, xhi + xpad], [ylo - ypad, yhi + ypad]],
        )
        h = gaussian_filter(h, sigma=2.0)
        lvls = post2d__contour_levels(h)
        xc = 0.5 * (xe[:-1] + xe[1:])
        yc = 0.5 * (ye[:-1] + ye[1:])
        contour_data.append((sigma_val, xc, yc, h, lvls))
        if sigma_val <= 0.003:
            all_qiso.append(qiso)
            all_qap.append(qap)
    zoom = None
    if all_qiso:
        qi = np.concatenate(all_qiso)
        qa = np.concatenate(all_qap)
        xlo, xhi = np.percentile(qi, [0.5, 99.5])
        ylo, yhi = np.percentile(qa, [0.5, 99.5])
        padx = 0.2 * (xhi - xlo)
        pady = 0.2 * (yhi - ylo)
        zoom = (xlo - padx, xhi + padx, ylo - pady, yhi + pady)
    return contour_data, zoom


def joint_gaussian_ellipse(mean, covariance, nsigma: float, points: int = 361) -> np.ndarray:
    """Points of the Gaussian contour at Mahalanobis radius *nsigma* (39.3 % / 86.5 % for 1 / 2)."""
    cov = np.asarray(covariance, dtype=float)
    values, vectors = np.linalg.eigh(cov)
    if np.any(values <= 0):
        raise ValueError("joint covariance must be positive definite")
    angle = np.linspace(0, 2 * np.pi, points)
    circle = np.stack([np.cos(angle), np.sin(angle)])
    return (np.asarray(mean, dtype=float)[:, None]
            + nsigma * vectors @ (np.sqrt(values)[:, None] * circle)).T


def post2d__draw_joint(ax, name: str) -> None:
    """Overlay the joint-constraint Gaussian at 1 and 2 sigma where the bin has a joint record."""
    joint = joint_record_for(name)
    if joint is None:
        return
    for nsigma, lw in ((2.0, 0.8), (1.0, 1.2)):
        points = joint_gaussian_ellipse(joint["mean"], joint["covariance"], nsigma)
        ax.plot(points[:, 0], points[:, 1], color=JOINT_COLOR, lw=lw, ls="--", zorder=5)


def post2d__draw_panel(ax, contour_data) -> None:
    """Draw the σ-tier contours + fiducial crosshair on one panel."""
    for sigma_val, xc, yc, h, lvls in contour_data:
        ax.contour(
            xc,
            yc,
            h.T,
            levels=lvls,
            colors=[sigma_color(sigma_val)],
            linewidths=[0.8, 1.2],
        )
    ax.axhline(1.0, color="grey", lw=0.4, linestyle=":", zorder=1)
    ax.axvline(1.0, color="grey", lw=0.4, linestyle=":", zorder=1)


def post2d__legend(legend_ax) -> None:
    handles = []
    for sigma in _OVERLAY_SIGMAS:
        handles.append(plt.Line2D([0], [0], color=sigma_color(sigma), label=rf"$\sigma_0={sigma:g}$"))
    handles.append(plt.Line2D([0], [0], color=JOINT_COLOR, ls="--", lw=1.2,
                              label="joint (four classes), Gaussian"))
    legend_ax.legend(
        handles,
        [h.get_label() for h in handles],
        loc="center",
        frameon=False,
        fontsize="small",
    )


if BASELINE_REPORT is not None:
    plt.style.use(PRD_STYLE_FILE)
    panels = {(t, z0, z1): post2d__collect_panel(t, z0, z1) for t, z0, z1, _l in ZBIN_GRID_PANELS}

    # Primary: identical [0.95, 1.05]² view on all five panels (shared axes),
    # so posterior sizes compare directly across z-bins. The broad σ=0.01
    # and σ=0.03 contours clip out of this common view.
    fig, axes = zbin_grid_axes(
        figsize_kind="two_col",
        h=5.0,
        xlabel=r"$\alpha_{\rm iso}$",
        ylabel=r"$\alpha_{\rm AP}$",
    )
    for ax, name, (tracer, zmin, zmax, _label) in zip(axes[:5], BINS, ZBIN_GRID_PANELS):
        post2d__draw_panel(ax, panels[(tracer, zmin, zmax)][0])
        post2d__draw_joint(ax, name)
        zbin_panel_label(ax, tracer, zmin, zmax)
    axes[0].set_xlim(0.95, 1.05)
    axes[0].set_ylim(0.95, 1.05)
    post2d__legend(axes[5])
    show(fig, "fig_post2d")


## Reconstruction quality

Paired per-mock parabolic-error ratios retain the original bootstrap and spread. The diagonal FoM proxy is labeled separately from the joint covariance determinant. Missing BGS individual fits remain blank and are identified in the caption.


In [ ]:
def _gain_1d(fits: dict, pre_key: tuple, post_key: tuple) -> float:
    """1D recon gain σ_pre(α_iso)/σ_post(α_iso), or NaN if missing."""
    if pre_key not in fits or post_key not in fits:
        return np.nan
    pre = fits[pre_key].get("qiso_std", np.nan)
    post = fits[post_key].get("qiso_std", np.nan)
    if np.isfinite(pre) and np.isfinite(post) and post > 0:
        return pre / post
    return np.nan


def _gain_2d(fits: dict, pre_key: tuple, post_key: tuple) -> float:
    """Gain in diagonal precision 1/[sigma_iso sigma_AP], excluding correlation."""
    if pre_key not in fits or post_key not in fits:
        return np.nan
    vals = (
        fits[pre_key].get("qiso_std", np.nan),
        fits[pre_key].get("qap_std", np.nan),
        fits[post_key].get("qiso_std", np.nan),
        fits[post_key].get("qap_std", np.nan),
    )
    si_pre, sa_pre, si_post, sa_post = vals
    if all(np.isfinite(v) and v > 0 for v in vals):
        return (si_pre * sa_pre) / (si_post * sa_post)
    return np.nan


from pzbao.figures.data import joint_recon_gain_distribution, recon_gain_distribution

def recon_quality_figure(gain_rows):
    """Draw the paired-gain figure from its rows (also the saved-rows replot path)."""
    summaries = {(r["measurement_bin"], r["condition"], r["metric"]): r["summary"]
                 for r in gain_rows}
    plt.style.use(PRD_STYLE_FILE)
    # DEFAULT (paper figure fig_recon_quality): the main 10-subplot quality plot
    # -- one panel per z-bin (columns) x {1D, 2D} (rows), gain vs sigma_z, with
    # a common y-axis per row (styled like fig_recon_quality_uncertainty_grid).
    # Filled markers = per-mock median gain with the bootstrap 68% CI error bar
    # (as in the main fig_recon_quality). The dotted line marks unit gain; the
    # 16-84% per-mock spread stays in the supporting table, not the figure.
    fig, axes = plt.subplots(
        2, 5, sharex=True, sharey="row", figsize=wide(h=4.4), layout="constrained"
    )
    band_rows = (
        (
            "1d",
            _gain_1d,
            r"$\sigma_{\rm pre}(\alpha_{\rm iso})/\sigma_{\rm post}(\alpha_{\rm iso})$",
        ),
        (
            "2d",
            _gain_2d,
            r"$\mathrm{FoM}_{\rm 2D,diag}^{\rm post}/\mathrm{FoM}_{\rm 2D,diag}^{\rm pre}$",
        ),
    )
    row_extent = {0: [1.0], 1: [1.0]}
    sig4 = ("sigma_0.001", "sigma_0.003", "sigma_0.01", "sigma_0.03")
    # The four-condition combination of each realization sits to the right of the conditions.
    JOINT_X = 0.07
    xticks = [0.001, 0.003, 0.01, 0.03, JOINT_X]
    xticklabels = ["0.001", "0.003", "0.01", "0.03", "joint"]
    for row, (kind, gain, ylab) in enumerate(band_rows):
        for col, (tracer, zmin, zmax, label) in enumerate(ZBIN_GRID_PANELS):
            ax = axes[row, col]
            ax.axhline(1.0, color="grey", lw=0.6, linestyle=":", zorder=1)
            # The joint point combines the four classes, so a dashed line at the geometric
            # midpoint of the log axis sets it apart from them.
            ax.axvline(np.sqrt(xticks[-2] * JOINT_X), color="0.5", lw=0.8, ls="--", zorder=1)
            for sl in sig4:
                sv = SIGMA_LABEL_TO_VALUE[sl]
                c = sigma_color(sv)
                dist = summaries[(BINS[col], sl, kind)]
                if dist is None:
                    continue
                ax.errorbar(
                    sv,
                    dist["median"],
                    yerr=[[dist["median"] - dist["boot_lo"]], [dist["boot_hi"] - dist["median"]]],
                    fmt="o",
                    ms=4,
                    color=c,
                    capsize=2.5,
                    elinewidth=1.4,
                    zorder=4,
                )
                row_extent[row] += [dist["boot_lo"], dist["boot_hi"]]
            joint = summaries[(BINS[col], "joint", kind)]
            if joint is not None:
                ax.errorbar(
                    JOINT_X,
                    joint["median"],
                    yerr=[[joint["median"] - joint["boot_lo"]], [joint["boot_hi"] - joint["median"]]],
                    fmt="D",
                    ms=4,
                    color="k",
                    capsize=2.5,
                    elinewidth=1.4,
                    zorder=4,
                )
                row_extent[row] += [joint["boot_lo"], joint["boot_hi"]]
            ax.set_xscale("log")
            ax.set_xlim(7e-4, 1.0e-1)
            ax.set_xticks(xticks)
            ax.xaxis.set_minor_locator(plt.NullLocator())  # categories: major ticks only
            ax.grid(True, which="major", alpha=0.25)
            if row == 0:
                ax.set_title(label, fontsize="xx-small")
            if row == 1:
                ax.set_xticklabels(xticklabels, fontsize="xx-small", rotation=45)
                ax.set_xlabel(r"$\sigma_z/(1+z)$", fontsize="x-small")
        axes[row, 0].set_ylabel(ylab, fontsize="x-small")
    for row, extent in row_extent.items():
        low, high = np.nanmin(extent), np.nanmax(extent)
        pad = 0.08 * (high - low)
        axes[row, 0].set_ylim(low - pad, high + pad)
    band_handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="k",
            ls="none",
            ms=4,
            label="Median + bootstrap 68% interval",
        ),
        plt.Line2D(
            [0], [0], marker="D", color="k", ls="none", ms=4, label="Joint of the four classes"
        ),
    ]
    fig.legend(
        handles=band_handles, loc="outside upper center", ncol=2, frameon=False, fontsize=8
    )
    show(fig, "fig_recon_quality")

if BASELINE_REPORT is not None:
    sig4 = ("sigma_0.001", "sigma_0.003", "sigma_0.01", "sigma_0.03")
    gain_rows = []
    for kind in ("1d", "2d"):
        for col, (tracer, zmin, zmax, label) in enumerate(ZBIN_GRID_PANELS):
            for sl in sig4:
                dist = recon_gain_distribution(profiles_full, tracer, zmin, zmax, sl, 15, kind=kind)
                gain_rows.append(dict(measurement_bin=BINS[col], condition=sl, metric=kind, summary=dist))
            joint = joint_recon_gain_distribution(
                profiles_full, joint_rows, BINS[col], tracer, zmin, zmax, 15, kind=kind
            )
            gain_rows.append(dict(measurement_bin=BINS[col], condition="joint", metric=kind, summary=joint))
    supporting_tables["paired-gains"] = gain_rows
    recon_quality_figure(gain_rows)
elif globals().get("saved_baseline_rows") is not None:
    recon_quality_figure(saved_baseline_rows["paired-gains"])

In [ ]:
from pzbao.diagnostics.fitting import detection_statistics


def draw_detection(ax, mean_fits, fits):
    """Draw the bowl diagnostic into a square axis without calling it a likelihood scan."""
    tracer, zmin, zmax, _ = ZBIN_GRID_PANELS[BINS.index(DETECTION_BIN)]
    alpha = np.linspace(0.8, 1.2, 500)
    detection_rows = []
    for label, sigma in SIGMA_LABEL_TO_VALUE.items():
        key = (tracer, zmin, zmax, label, 15, "postrecon")
        saved, posterior = mean_fits[key], fits[key]
        stats = detection_statistics(saved["chi2"], saved["detection"]["chi2_smooth"])
        detection_rows.append(dict(measurement_bin=DETECTION_BIN, condition=label,
                                   reconstruction_state="postreconstruction", **stats))
        color = sigma_color(sigma)
        mean, std = posterior["qiso_mean"], posterior["qiso_std"]
        ax.plot(alpha, ((alpha - mean) / std) ** 2, color=color, lw=1.1)
        ax.plot(alpha, np.full_like(alpha, stats["delta_chi2"]), color=color, lw=0.9, ls="-.")
    ymax = max(155.0, 1.12 * max(row["delta_chi2"] for row in detection_rows))
    ax.axvline(1.0, color="0.7", lw=0.5, ls=":")
    ax.set(xlim=(0.8, 1.2), ylim=(0, ymax), xlabel=r"$\alpha_{\rm iso}$", ylabel=r"$\Delta\chi^2$")
    ax.set_box_aspect(1)
    ax.grid(False)
    supporting_tables["detection"] = detection_rows
    return detection_rows

## Correlation and paired residuals

These are the saved pooled-count means and full mock-to-mock standard deviations, including paired residual scatter. All displayed coordinates are retained.


In [ ]:
def lrg1_correlation_figure(overlays, mean_fits, fits):
    """Pre/post pooled monopoles for the five tiers; the detection bowls fill slot six."""
    fig = plt.figure(figsize=two_col(5.4))
    outer = fig.add_gridspec(2, 3, left=0.075, right=0.945, bottom=0.075, top=0.915,
                             wspace=0.08, hspace=0.30)
    mains, residuals = [], []
    for index, (condition, sigma) in enumerate(zip(TIERS, SIGMA_LABEL_TO_VALUE.values())):
        inner = outer[index // 3, index % 3].subgridspec(2, 1, height_ratios=(3, 1), hspace=0.05)
        ax = fig.add_subplot(inner[0], sharey=mains[0] if mains else None)
        resid = fig.add_subplot(inner[1], sharex=ax, sharey=residuals[0] if residuals else None)
        overlay = overlays[condition]
        for state, short, ls in zip(STATES, ("prerecon", "postrecon"), ("--", "-")):
            row = overlay[state]
            color = sigma_color(sigma, short)
            ax.plot(*row["curve"].T, color=color, ls=ls, lw=1)
            ax.errorbar(*row["markers"].T, yerr=row["error"], fmt="o", color=color,
                        mfc="white" if short == "prerecon" else color, ms=2.4,
                        elinewidth=0.6, capsize=1.2)
        resid.plot(*overlay["residual"].T, color=sigma_color(sigma), lw=0.9)
        resid.axhline(0, color="0.5", ls=":", lw=0.6)
        ax.text(0.95, 0.93, rf"$\sigma_0={sigma:g}$", transform=ax.transAxes, ha="right",
                va="top", color=sigma_color(sigma))
        ax.tick_params(labelbottom=False)
        if index % 3:
            ax.tick_params(labelleft=False)
            resid.tick_params(labelleft=False)
        else:
            ax.set_ylabel(r"$s^2\xi_0(s)\ [(h^{-1}{\rm Mpc})^2]$")
            resid.set_ylabel(r"$\Delta\xi_0/\sigma_{\rm post}$")
        if index >= 2:
            resid.set_xlabel(r"$s\ [h^{-1}{\rm Mpc}]$")
        for axis in (ax, resid):
            axis.grid(False)
        mains.append(ax)
        residuals.append(resid)
    mains[0].set_xlim(45, 155)
    bowl = fig.add_subplot(outer[1, 2])
    draw_detection(bowl, mean_fits, fits)
    bowl.yaxis.set_label_position("right")
    bowl.yaxis.tick_right()
    _, zmin, zmax, _ = ZBIN_GRID_PANELS[BINS.index(DETECTION_BIN)]
    bowl.set_title(rf"${zmin:g}\leq z<{zmax:g}$, post-reconstruction", pad=3)
    pre, post = (sigma_color(0, state) for state in ("prerecon", "postrecon"))
    fig.legend(handles=[
        plt.Line2D([], [], color=pre, ls="--", marker="o", mfc="white", ms=3,
                   label="Pre-reconstruction"),
        plt.Line2D([], [], color=post, ls="-", marker="o", ms=3, label="Post-reconstruction"),
        plt.Line2D([], [], color="0.3", lw=1.1, label="Gaussian posterior approximation"),
        plt.Line2D([], [], color="0.3", lw=0.9, ls="-.",
                   label=r"$\chi^2_{\rm smooth}-\chi^2_{\rm BAO}$"),
    ], loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=4, frameon=False, fontsize=8)
    return fig


if BASELINE_REPORT is not None:
    if SELECTION_RUN is None:
        raise ValueError("Pre/post monopoles require the matched frozen selection run")
    if OVERLAY_BIN != DETECTION_BIN:
        raise ValueError("The combined figure shows one measurement bin")
    lrg1_overlays = {
        condition: readers.load_reconstruction_overlay(
            SELECTION_RUN, measurement_bin=OVERLAY_BIN, condition=condition,
            expected_identity=baseline["measurement_identity"], sources=baseline["sources"],
        )
        for condition in TIERS
    }
    plt.style.use(PRD_STYLE_FILE)
    fig = lrg1_correlation_figure(lrg1_overlays, mean_fits, fits)
    supporting_tables["correlation"] = lrg1_overlays
    show(fig, "fig_cf")

In [ ]:
def lrg1_realizations_figure(profiles, measurement_bin="LRG1"):
    """Individual-mock fits per tier, after Paillas et al. (2024) Fig. 13.

    Each block holds the ensemble column and mocks 0-24; open/filled markers are pre/post.
    Errors are native Minos half-widths; unusable intervals (invalid or at a prior
    boundary) are marked with crosses and excluded from every summary.
    """
    tracer, zmin, zmax, _ = ZBIN_GRID_PANELS[BINS.index(measurement_bin)]
    rows = (("qiso_bf", r"$\alpha_{\rm iso}$"), ("qap_bf", r"$\alpha_{\rm AP}$"),
            ("qiso_minos", r"$\sigma(\alpha_{\rm iso})$"), ("qap_minos", r"$\sigma(\alpha_{\rm AP})$"),
            ("chi2", r"$\chi^2$"))
    width = 28  # ensemble column, 25 mocks and a two-column gap
    fig, axes = plt.subplots(5, 1, sharex=True, figsize=two_col(6.6),
                             gridspec_kw=dict(hspace=0.07, left=0.085, right=0.995,
                                              bottom=0.06, top=0.93))
    summary, dofs = [], set()
    for block, (label, sigma) in enumerate(SIGMA_LABEL_TO_VALUE.items()):
        x0 = block * width
        for state, offset in (("prerecon", -0.18), ("postrecon", 0.18)):
            members = profiles[(tracer, zmin, zmax, label, 15, state)]
            mocks = sorted(members)
            color = sigma_color(sigma, state)
            filled = dict(mfc=color if state == "postrecon" else "white", mec=color, mew=0.9)
            best = {key: np.array([members[m][key] for m in mocks], dtype=float)
                    for key, _ in rows}
            dofs |= {members[m]["dof"] for m in mocks}
            optimizer = np.array([members[m]["valid"] for m in mocks])
            x = x0 + 1 + np.arange(len(mocks)) + offset
            for row, (key, _) in enumerate(rows):
                ax = axes[row]
                values = np.where(optimizer, best[key], np.nan)
                usable = optimizer.copy()
                if key.endswith("_bf"):
                    error = best[key.replace("_bf", "_minos")]
                    usable &= np.isfinite(error)
                    ax.errorbar(x[usable], values[usable], yerr=error[usable], fmt="o", ms=2.6,
                                color=color, elinewidth=0.6, capsize=0, **filled)
                    ax.plot(x[optimizer & ~usable], values[optimizer & ~usable], "x", ms=3,
                            color=color, mew=0.8)
                else:
                    usable &= np.isfinite(values)
                    ax.plot(x[usable], values[usable], "o", ms=2.6, color=color, **filled)
                kept = values[usable]
                mean = float(kept.mean()) if kept.size else None
                std = float(kept.std(ddof=1)) if kept.size > 1 else None
                if mean is not None:
                    ax.errorbar(x0 + offset, mean, yerr=std if key.endswith("_bf") else None,
                                fmt="s", ms=3.4, color=color, elinewidth=0.8, capsize=0, **filled)
                summary.append(dict(measurement_bin=measurement_bin, condition=label,
                                    state=state, quantity=key, mean=mean, std=std,
                                    count=int(kept.size)))
            if state == "postrecon":
                for row, key in ((0, "qiso_bf"), (1, "qap_bf")):
                    fit = best[key][optimizer & np.isfinite(best[key.replace("_bf", "_minos")])]
                    if fit.size < 2:
                        continue
                    center, scatter = fit.mean(), fit.std(ddof=1)
                    axes[row].fill_between([x0 - 0.6, x0 + 25.6], center - scatter, center + scatter,
                                           color=sigma_color(sigma, "prerecon"), lw=0, zorder=0)
                    axes[row + 2].plot([x0 - 0.6, x0 + 25.6], [scatter, scatter], color=color,
                                       ls="--", lw=0.9, zorder=0)
        for ax in axes:
            ax.axvline(x0 + 0.5, color="0.85", lw=0.5, zorder=0)
    (dof,) = dofs
    axes[4].axhline(dof, color="0.4", ls=":", lw=0.8, zorder=0)
    for ax, (key, ylabel) in zip(axes, rows):
        ax.set_ylabel(ylabel + (rf" (dof $={dof}$)" if key == "chi2" else ""))
        ax.grid(False)
    for ax in axes[:2]:
        ax.axhline(1.0, color="0.4", ls=":", lw=0.8, zorder=0)
    for ax in axes[2:4]:
        ax.set_yscale("log")
    centers = [block * width + 12.5 for block in range(len(SIGMA_LABEL_TO_VALUE))]
    axes[-1].set_xticks(centers)
    axes[-1].set_xticklabels([rf"$\sigma_0={s:g}$" for s in SIGMA_LABEL_TO_VALUE.values()])
    axes[-1].tick_params(axis="x", which="both", length=0)
    axes[-1].xaxis.set_minor_locator(plt.NullLocator())  # categories: major ticks only
    axes[-1].set_xlim(-1.5, len(centers) * width - 1.5)
    for tick, sigma in zip(axes[-1].get_xticklabels(), SIGMA_LABEL_TO_VALUE.values()):
        tick.set_color(sigma_color(sigma))
    fig.legend(handles=[
        plt.Line2D([], [], ls="none", marker="o", mfc="white", mec="0.35", ms=3.5,
                   label="Pre-reconstruction"),
        plt.Line2D([], [], ls="none", marker="o", color="0.35", ms=3.5, label="Post-reconstruction"),
        plt.Line2D([], [], ls="none", marker="s", color="0.35", ms=3.5,
                   label="Mean of 25 fits"),
        plt.Line2D([], [], ls="none", marker="x", color="0.35", ms=3.5,
                   label="Unusable interval"),
        Patch(color="0.85", label="Post: mean $\\pm$ s.d. of best fits"),
        plt.Line2D([], [], color="0.35", ls="--", lw=0.9, label="Post: s.d. of best fits"),
    ], loc="upper center", ncol=6, frameon=False, fontsize=7, handletextpad=0.3,
               columnspacing=0.9)
    supporting_tables["lrg1-realizations"] = summary
    return fig


if BASELINE_REPORT is not None:
    plt.style.use(PRD_STYLE_FILE)
    show(lrg1_realizations_figure(profiles_full), "fig_lrg1_realizations")

In [ ]:
# The LRG1 pre/post overlay figure (former Fig. 4) was retired on 2026-10-15; its overlays
# still feed the correlation figure (fig_cf) through lrg1_overlays.


In [ ]:
TIER_COLORS = tuple(sigma_color(value) for value in SIGMA_LABEL_TO_VALUE.values())
# Colors read from the SVG of Paillas et al. 2404.03005v3, Figure 15.
BIN_COLORS = dict(zip(BINS, ("#9acd32", "#9acd32", "#ffa500", "#ff4500", "#b22222")))
ZLABELS = (
    r"$0.1\leq z<0.2$",
    r"$0.2\leq z<0.4$",
    r"$0.4\leq z<0.6$",
    r"$0.6\leq z<0.8$",
    r"$0.8\leq z<1.0$",
)
VARIANTS = ("baseline", "pre", "pk", "ngc", "flat", "pcs2", "reciso", "r25")
VARIANT_LABELS = (
    "Baseline: power3, RecSym, R15",
    "Pre-reconstruction",
    "Power spectrum only",
    "NGC only",
    "Flat damping priors",
    "PCS2 spline broadband",
    "RecIso",
    r"$R=25\,h^{-1}\mathrm{Mpc}$",
)
LABELS = {"qiso": r"$\alpha_{\rm iso}$", "qap": r"$\alpha_{\rm AP}$"}


def presentation_style():
    import matplotlib as mpl

    mpl.rcParams.update(
        {
            "font.family": "serif",
            "mathtext.fontset": "cm",
            "font.size": 8,
            "axes.labelsize": 9,
            "axes.titlesize": 9,
            "axes.linewidth": 0.8,
            "xtick.direction": "in",
            "ytick.direction": "in",
            "xtick.top": True,
            "ytick.right": True,
            "savefig.dpi": 200,
        }
    )


def panel_grid():
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 3, figsize=(13.5, 8), layout="constrained")
    axes = list(axes.flat)
    axes[-1].axis("off")
    return fig, axes


presentation_style()


In [ ]:
FOM_TIERS = tuple(tier for tier in TIERS if tier != "native")  # the unsmeared control is a reference


def joint_fom_pairs(joint_rows):
    """Pre/post joint FoM per bin from the combination records, in the fom-values row format."""
    pairs = []
    for name in BINS:
        by_state = {
            r["route"]["reconstruction_state"]: r
            for r in joint_rows
            if r["route"]["measurement_bin"] == name
        }
        if set(by_state) != set(STATES):
            raise ValueError(f"joint records incomplete for {name}")
        if any(by_state[state]["status"] != "available" for state in STATES):
            continue  # no per-realization profiles for this bin: no joint bar, shown as n/a
        pre, post = (by_state[state]["fom"] for state in STATES)
        pairs.append(
            dict(measurement_bin=name, condition="joint", pre=pre, post=post,
                 gain_percent=100 * (post / pre - 1))
        )
    return pairs


def presentation_plot_fom(pairs, joint_rows):
    """Combine the tier records of each bin into joint pairs, save them, and draw the bars."""
    joint_pairs = joint_fom_pairs(joint_rows)
    supporting_tables["fom-joint"] = joint_pairs
    fom_bar_figure(pairs, joint_pairs)


def fom_bar_figure(pairs, joint_pairs):
    """Draw the FoM bars from the tier pairs and the joint pairs (also the saved-rows replot path)."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    from matplotlib.ticker import FuncFormatter, MaxNLocator

    pairs = [r for r in pairs if r["condition"] != "native"] + list(joint_pairs)
    columns = (*FOM_TIERS, "joint")
    fig = plt.figure(figsize=two_col(6.0))
    grid = fig.add_gridspec(
        2, 3, left=0.09, right=0.99, bottom=0.09, top=0.94, wspace=0.48, hspace=0.42
    )
    for index, (name, zlabel) in enumerate(zip(BINS, ZLABELS)):
        group = {r["condition"]: r for r in pairs if r["measurement_bin"] == name}
        if not set(FOM_TIERS) <= set(group) or not set(group) <= set(columns):
            raise ValueError(f"FoM rows incomplete for {name}")
        ax = fig.add_subplot(grid[index // 3, index % 3])
        largest = max(max(row["pre"], row["post"]) for row in group.values())
        # One linear axis per panel; no broken axes.
        ax.set_ylim(0, largest * 1.30)
        ax.set_xlim(-0.6, len(columns) - 0.4)
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
        ax.yaxis.set_major_locator(MaxNLocator(5))
        ax.set_axisbelow(True)
        ax.grid(axis="y", color=".9")
        ax.minorticks_on()
        ax.xaxis.set_minor_locator(plt.NullLocator())  # categories: major ticks only
        # The joint bar combines the four classes, so a dashed line sets it apart from them.
        ax.axvline(len(FOM_TIERS) - 0.5, color="0.5", lw=0.8, ls="--", zorder=0)
        for x, condition in enumerate(columns):
            if condition not in group:
                ax.text(x, 0.03 * largest, "n/a", ha="center", va="bottom", fontsize=6.5,
                        color="0.4", gid="gain-label")
                continue
            if condition == "joint":
                color = JOINT_COLOR
            else:
                color = sigma_color(float(condition.split("_")[1]), "postrecon")
            row = group[condition]
            # Same colour in both states; black hatching marks pre-reconstruction. The taller bar
            # is drawn first so a loss does not hide the post-reconstruction bar.
            for state in sorted(("post", "pre"), key=lambda state: -row[state]):
                if state == "pre":
                    ax.bar(x, row["pre"], 0.75, facecolor=color, edgecolor="black", hatch="///",
                           linewidth=0)
                else:
                    ax.bar(x, row["post"], 0.75, color=color, linewidth=0)
                ax.bar(x, row[state], 0.75, fill=False, edgecolor="black", linewidth=0.8)
            top = max(row["pre"], row["post"])
            ax.annotate(
                f"{row['gain_percent']:+.1f}%", (x, top), xytext=(0, 4),
                textcoords="offset points", ha="center", va="bottom", fontsize=6.5,
                gid="gain-label", zorder=2.2,  # above the dashed line, below the axis spines
                bbox=dict(boxstyle="square,pad=0.1", facecolor="white", edgecolor="none"),
            )
        ax.set_title(zlabel)
        ax.set(
            xticks=range(len(columns)),
            xticklabels=("0.001", "0.003", "0.01", "0.03", "joint"),
            xlabel=r"$\sigma_z/(1+z)$",
        )
    fig.text(0.012, 0.5,
             r"$\mathrm{FoM}_{\rm 2D}=1/\sqrt{\det\,\mathrm{Cov}(\alpha_{\rm iso},\alpha_{\rm AP})}$",
             rotation=90, ha="left", va="center", fontsize=9)
    legend = fig.add_subplot(grid[1, 2])
    legend.axis("off")
    handles = [
        Patch(facecolor=sigma_color(s, "postrecon"), edgecolor="black",
              label=rf"$\sigma_z/(1+z)={t}$")
        for s, t in zip(SIGMA_TIERS, ("0.001", "0.003", "0.01", "0.03"))
    ]
    handles.append(Patch(facecolor=JOINT_COLOR, edgecolor="black", label="Joint of the four classes"))
    handles += [
        Patch(facecolor="white", edgecolor="black", label="Post-reconstruction"),
        (Patch(facecolor="white", edgecolor="black", hatch="///", linewidth=0),
         Patch(fill=False, edgecolor="black")),
    ]
    labels = [h.get_label() for h in handles[:-1]] + ["Pre-reconstruction"]
    legend.legend(handles=handles, labels=labels, loc="center left", frameon=False, fontsize=8)
    show(
        fig,
        "fig_fom_2d",
        "FoM = 1/sqrt(det Cov(alpha_iso,alpha_AP)), ddof=1, for the four redshift precision classes and "
        "their joint combination (non-negative combination of the four tier posteriors, "
        "specification revision 2.14). Solid post-reconstruction bars and black-hatched "
        "pre-reconstruction bars in the same colour; signed percentage gains. One linear axis "
        "per panel; the unsmeared control is not shown; n/a marks bins whose designated report "
        "carries no per-realization profiles for the joint combination. All saved posteriors retain sampling "
        "limitations; a profile flag does not replace posterior diagnostics.",
    )


if BASELINE_REPORT is not None:
    presentation_style()
    presentation_plot_fom(fom_pairs, joint_rows)
elif globals().get("saved_baseline_rows") is not None:
    presentation_style()
    fom_bar_figure(saved_baseline_rows["fom-values"], saved_baseline_rows["fom-joint"])


In [ ]:
def presentation_plot_bias(rows, full=False):
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    group = [r for r in rows if r["parameter"] == "qiso"]
    limits = (0.8, 1.2)
    if full:
        lo = min([0.8] + [r["mean"] - r["sd"] for r in group])
        hi = max([1.2] + [r["mean"] + r["sd"] for r in group])
        limits = (lo - 0.04 * (hi - lo), hi + 0.04 * (hi - lo))
    fig, axes = plt.subplots(1, 5, figsize=(14, 5.7), sharey=True, layout="constrained")
    for ax, name, zlabel in zip(axes, BINS, ZLABELS):
        for y, (tier, color) in enumerate(zip(TIERS, TIER_COLORS)):
            for state, offset in zip(STATES, (-0.11, 0.11)):
                row = next(
                    r
                    for r in group
                    if (r["measurement_bin"], r["condition"], r["reconstruction_state"])
                    == (name, tier, state)
                )
                mean, sd = row["mean"], row["sd"]
                yy = y + offset
                pre = state == STATES[0]
                ax.errorbar(
                    mean,
                    yy,
                    xerr=sd,
                    fmt="o",
                    color=color,
                    alpha=0.48 if pre else 1,
                    mfc="white" if pre else color,
                    mew=1.6,
                    ms=6,
                    capsize=3,
                    lw=1.6,
                )
                for bound, marker, side in ((limits[0], "<", -1), (limits[1], ">", 1)):
                    if mean - sd < bound if side == -1 else mean + sd > bound:
                        ax.plot(bound, yy, marker=marker, color=color, ms=6, clip_on=False)
                if not limits[0] <= mean <= limits[1]:
                    ax.text(
                        np.clip(mean, *limits), yy + 0.14, f"{mean:.3f}", ha="center", fontsize=8
                    )
        ax.axvline(1, color=".55", ls=":", lw=1.1)
        ax.set(title=f"{name}\n{zlabel}", xlim=limits, ylim=(4.5, -0.5), xlabel=LABELS["qiso"])
        if not full:
            ax.set_xticks([0.8, 0.9, 1, 1.1, 1.2])
        ax.tick_params(axis="x", labelsize=10)
        ax.minorticks_on()
    axes[0].set_yticks(range(5), (r"$\sigma_z=0$", "0.001", "0.003", "0.01", "0.03"))
    axes[0].set_ylabel(r"$\sigma_z/(1+z)$ sample")
    fig.legend(
        handles=[
            Line2D([], [], color=".5", marker="o", mfc="white", ls="", label="Pre-reconstruction"),
            Line2D([], [], color=".2", marker="o", ls="", label="Post-reconstruction"),
        ],
        loc="outside upper center",
        ncol=2,
        frameon=False,
    )
    suffix = "-fullrange" if full else ""
    show(
        fig,
        "F5-alpha-iso" + suffix,
        "Power3 mean-data posterior mean and one posterior SD; this is not ensemble bias. "
        "All sampling limitations remain. Edge triangles mark intervals or centers outside "
        "the zoom; the full-range companion retains every interval.",
    )


# Optional exploratory plot. The manuscript refresh retains the agreed core ranges
# and does not export an additional full-range posterior view.


In [ ]:
from pzbao.figures.style import PRE_STYLE, POST_STYLE


def render_diagnostic_power_demonstration(k_h_mpc, pre_multipoles, post_multipoles):
    """Draw pre/post ell=0,2,4 without measuring clustering or promoting manuscript assets."""
    k = np.asarray(k_h_mpc, dtype=float)
    before, after = np.asarray(pre_multipoles, dtype=float), np.asarray(
        post_multipoles, dtype=float
    )
    with plt.style.context(PRD_STYLE_FILE):
        figure, axes = plt.subplots(3, 1, sharex=True, figsize=two_col(7.2))
        for index, ell in enumerate((0, 2, 4)):
            axes[index].plot(k, before[index], **PRE_STYLE, label="pre-reconstruction")
            axes[index].plot(k, after[index], **POST_STYLE, label="post-reconstruction")
            axes[index].set_ylabel(f"P{ell}(k) [(Mpc/h)^3]")
        axes[0].legend()
        axes[-1].set_xlabel("k [h/Mpc]")
        figure.tight_layout()
        show(figure, "diagnostic-power")
    return figure


POWER_ARRAYS = None  # saved NPZ with k_h_mpc, pre_multipoles, post_multipoles
if POWER_ARRAYS is not None:
    arrays = readers.load_npz(POWER_ARRAYS)
    render_diagnostic_power_demonstration(
        arrays["k_h_mpc"], arrays["pre_multipoles"], arrays["post_multipoles"]
    )


In [ ]:
# Survey landscape as effective volume against redshift precision, on categorical columns.
# V_eff counts footprint volume and shot noise only; the radial-mode loss from photo-z
# errors is carried by the horizontal axis, not by V_eff.
#: Marker, colour and open face of each comparison survey; the saved rows carry only values.
SURVEY_MARKERS = {
    r"DESI DR1, $0.4<z<1.1$": ("s", style.categorical_color(0), False),
    r"DESI DR1, $0.1<z<0.4$": ("s", style.categorical_color(0), True),
    "LSST Y10": ("v", style.categorical_color(2), False),
    "Euclid photometric": ("^", style.categorical_color(1), False),
    "DES Y6": ("D", style.SURVEY_YELLOW, False),
}


def survey_landscape_figure(table):
    """Draw V_eff against redshift precision from the survey-veff table (also the saved-rows path)."""
    spherex, surveys, this_work = table["spherex"], table["surveys"], table["this_work"]
    columns = [column["sigma0"] for column in table["columns"]]
    position = {sigma: index for index, sigma in enumerate(columns)}
    occupancy = {}

    def dodge(sigma, slots):
        """Spread the markers sharing a column over small offsets, in plotting order."""
        taken = occupancy.setdefault(sigma, 0)
        occupancy[sigma] += 1
        return position[sigma] + (taken - (slots - 1) / 2) * 0.16

    slots = {sigma: sum(row["sigma0"] == sigma for row in spherex + surveys + this_work)
             for sigma in columns}
    this_work_color = style.DESI_FIREBRICK[1]
    filled_width, open_width = 0.5, 1.2

    def open_size(filled_size):
        """Area of an open marker whose outer diameter (sqrt(s) plus the edge width) matches the
        filled marker of area filled_size."""
        return (np.sqrt(filled_size) + filled_width - open_width) ** 2

    plt.style.use(PRD_STYLE_FILE)
    fig, ax = plt.subplots(figsize=style.one_col(3.2), layout="constrained")
    for index, row in enumerate(this_work):
        ax.scatter(dodge(row["sigma0"], slots[row["sigma0"]]), row["veff_gpc_h3"], s=42, marker="o",
                   color=this_work_color, edgecolors="black", linewidths=filled_width, zorder=8,
                   label="This work (mock samples)" if index == 0 else None)
    for index, row in enumerate(spherex):
        ax.scatter(dodge(row["sigma0"], slots[row["sigma0"]]), row["veff_gpc_h3"], s=open_size(42),
                   facecolors="white", edgecolors=this_work_color, linewidths=open_width, zorder=4,
                   label="SPHEREx galaxy samples" if index == 0 else None)
    for row in surveys:
        marker, color, opened = SURVEY_MARKERS[row["label"]]
        ax.scatter(dodge(row["sigma0"], slots[row["sigma0"]]), row["veff_gpc_h3"],
                   s=open_size(55) if opened else 55, marker=marker,
                   facecolors="white" if opened else color, edgecolors=color if opened else "black",
                   linewidths=open_width if opened else filled_width, zorder=6, label=row["label"])
    values = [r["veff_gpc_h3"] for r in spherex + surveys + this_work]
    ax.set_yscale("log")
    ax.set(xlim=(-0.6, len(columns) - 0.4), ylim=(min(values) / 2, max(values) * 2.5),
           xticks=range(len(columns)), xticklabels=[column["label"] for column in table["columns"]],
           xlabel=r"$\sigma_z/(1+z)$", ylabel=r"$V_{\rm eff}\ [(h^{-1}\,{\rm Gpc})^3]$")
    ax.xaxis.set_minor_locator(plt.NullLocator())  # categories: major ticks only
    ax.grid(True, axis="y", which="major", alpha=0.3)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False,
              fontsize=8, handletextpad=0.3, columnspacing=0.7)
    show(fig, "fig_survey_landscape")


if BASELINE_REPORT is not None:
    import yaml

    from pzbao.catalog import build_mock_cosmology

    if SELECTION_RUN is None:
        raise ValueError("Survey volumes require the frozen selection-run cosmology")
    cosmology = build_mock_cosmology(frozen_config)
    VEFF_P0 = 1.0e4  # h^-3 Mpc^3, one value for every survey
    FULL_SKY_DEG2 = 4 * np.pi * (180 / np.pi) ** 2
    SPECTROSCOPIC_LABEL = r"$10^{-5}$"  # spectroscopic precision column, as agreed

    survey_path = REPO / "pzbao/config/sources/survey_pars_v28_base_cbe.yaml"
    survey = yaml.safe_load(baseline["sources"].read_current(survey_path))
    edges = np.r_[survey["zbin_lo"], survey["zbin_hi"][-1]]
    # The repository table carries local edits to three sample-1 shells. Restore the public
    # v28 values from the frozen sigma_0.003 targets, which are those originals.
    tiers = {row["label"]: row for row in frozen_config["selection"]["tiers"]}
    public_sample1 = list(survey["number_density_in_hinvMpc_1"])
    for shell, name in ((2, "LRG1"), (3, "LRG2"), (4, "LRG3")):
        public_sample1[shell] = tiers["sigma_0.003"]["target_nbar_h3_mpc3"][name]
    for shell, name in ((0, "BGS1"), (1, "BGS2")):
        assert public_sample1[shell] == tiers["sigma_0.003"]["target_nbar_h3_mpc3"][name]
    # Reference sky fraction of the configuration: the SPHEREx effective area of 3x10^4 deg^2
    # (Feder et al. 2024, arXiv:2312.04636) over the full sky.
    configuration = yaml.safe_load(
        baseline["sources"].read_current(REPO / "pzbao/config/analysis.yaml"))
    SPHEREX_FSKY = float(configuration["reporting"]["forecast_target_fsky"])
    spherex = []
    for index, sigma in enumerate(survey["sigz_over_one_plus_z"], start=1):
        nbar = public_sample1 if index == 1 else survey[f"number_density_in_hinvMpc_{index}"]
        spherex.append(dict(
            sample=index, sigma0=sigma, z_edges=edges, nbar_h3_mpc3=list(nbar), f_sky=SPHEREX_FSKY,
            veff_gpc_h3=readers.survey_effective_volume(
                cosmology, edges, nbar, f_sky=SPHEREX_FSKY, p0_h3_mpc3=VEFF_P0),
        ))

    def uniform_survey(count, area_deg2, zmin, zmax):
        f_sky = area_deg2 / FULL_SKY_DEG2
        chi = cosmology.comoving_radial_distance(np.array([zmin, zmax]))
        nbar = count / (f_sky * 4 * np.pi / 3 * (chi[1] ** 3 - chi[0] ** 3))
        return readers.survey_effective_volume(
            cosmology, [zmin, zmax], [nbar], f_sky=f_sky, p0_h3_mpc3=VEFF_P0)

    def smail_survey(arcmin2_density, area_deg2, zmin, zmax, dndz):
        z_edges = np.linspace(zmin, zmax, int(round((zmax - zmin) / 0.01)) + 1)
        nbar = readers.surface_density_nbar(
            cosmology, z_edges, arcmin2_density=arcmin2_density, dndz=dndz)
        return readers.survey_effective_volume(
            cosmology, z_edges, nbar, f_sky=area_deg2 / FULL_SKY_DEG2, p0_h3_mpc3=VEFF_P0)

    euclid_z0 = 0.9 / np.sqrt(2)
    surveys = [
        dict(label=r"DESI DR1, $0.4<z<1.1$", sigma0=0.0,
             veff_gpc_h3=uniform_survey(2_138_600, 5840.0, 0.4, 1.1),
             inputs="2,138,600 redshifts, 5,840 deg2, 0.4<z<1.1",
             source="arXiv:2404.03000, Sec. 2.2 and Table 1"),
        dict(label=r"DESI DR1, $0.1<z<0.4$", sigma0=0.0,
             veff_gpc_h3=uniform_survey(300_017, 7473.0, 0.1, 0.4),
             inputs="300,017 redshifts, 7,473 deg2, 0.1<z<0.4",
             source="arXiv:2404.03000, Sec. 2.2 and Table 1"),
        dict(label="LSST Y10", sigma0=0.03,
             veff_gpc_h3=smail_survey(48.0, 14_300.0, 0.2, 1.2,
                                      lambda z: z**2 * np.exp(-(z / 0.28) ** 0.90)),
             inputs="lens sample 48 arcmin-2, 14.3k deg2, dN/dz ~ z^2 exp[-(z/0.28)^0.90], 0.2<z<1.2",
             source="arXiv:1809.01669, Appendices C1 and D1"),
        dict(label="Euclid photometric", sigma0=0.05,
             veff_gpc_h3=smail_survey(30.0, 15_000.0, 0.001, 2.5,
                                      lambda z: (z / euclid_z0) ** 2 * np.exp(-(z / euclid_z0) ** 1.5)),
             inputs="30 arcmin-2, 15,000 deg2, n(z) ~ (z/z0)^2 exp[-(z/z0)^1.5], z0=0.9/sqrt2, 0.001<z<2.5",
             source="arXiv:1910.09273, Table 4 and Eqs. 113-114"),
        dict(label="DES Y6", sigma0=0.03,
             veff_gpc_h3=uniform_survey(15_937_556, 4273.42, 0.6, 1.2),
             inputs="15,937,556 galaxies, 4,273.42 deg2, 0.6<z_ph<1.2, ~3% photo-z precision",
             source="arXiv:2402.10696, Sec. II"),
    ]
    this_work = []
    for condition, sigma in zip(TIERS[1:], SIGMA_TIERS):  # the unsmeared control is not a survey
        caps = [cap for row in precision_rows if row["condition"] == condition for cap in row["caps"]]
        assert len(caps) == 2 * len(BINS)
        veff = readers.effective_volume([c["volume_mpc_h3"] for c in caps],
                                        [c["nbar_h3_mpc3"] for c in caps], p0_h3_mpc3=VEFF_P0)
        this_work.append(dict(condition=condition, sigma0=sigma, veff_gpc_h3=float(veff.sum() / 1e9),
                              inputs="attained density per measurement bin and cap, 0.1<z<1.0"))

    # Categorical columns: one per distinct redshift precision; spectroscopic surveys share one.
    columns = sorted({row["sigma0"] for row in spherex + surveys + this_work})
    supporting_tables["survey-veff"] = dict(
        definition="V_eff = sum_shells f_sky V_shell [nbar P0/(1+nbar P0)]^2",
        p0_h3_mpc3=VEFF_P0, spherex=spherex, this_work=this_work,
        columns=[dict(sigma0=sigma, label=SPECTROSCOPIC_LABEL if sigma == 0.0 else f"{sigma:g}")
                 for sigma in columns],
        spherex_source="SPHEREx/Public-products galaxy_density_v28_base_cbe.txt; f_sky = "
                       "reporting.forecast_target_fsky (30,000 deg^2, arXiv:2312.04636)",
        surveys=surveys,
        interpretation="Volume and shot noise only; photo-z radial smearing is not included; "
                       "columns are categorical redshift-precision classes",
    )
    survey_landscape_figure(supporting_tables["survey-veff"])
elif globals().get("saved_baseline_rows") is not None:
    survey_landscape_figure(saved_baseline_rows["survey-veff"])


In [ ]:
# These figures describe the designated baseline and its analytic response.
from pzbao.catalog import build_mock_cosmology

# Analytic photo-z propagator of the galaxy field, G = exp[-(k mu sigma_r)^2 / 2], averaged over
# three equal line-of-sight wedges, for the two redshift ranges at each tier's attained effective
# redshift (count-weighted over the range's measurement bins). Measured points follow from the
# propagator diagnostic once the per-bin successors publish.
RANGES = (("0.1 \\leq z < 0.4", ("BGS1", "BGS2")), ("0.4 \\leq z < 1.0", ("LRG1", "LRG2", "LRG3")))
WEDGES = ((0.0, 1 / 3), (1 / 3, 2 / 3), (2 / 3, 1.0))
WEDGE_STYLES = ("-", "--", ":")


def los_damping_figure(table, drag_scale_mpc_h):
    """G(k) per redshift range, tier and |mu| wedge, with the BAO wavenumber k = 2 pi / r_d."""
    plt.style.use(PRD_STYLE_FILE)
    k_bao = 2 * np.pi / drag_scale_mpc_h
    fig, axes = plt.subplots(2, 1, figsize=style.one_col(4.4), sharex=True, layout="constrained")
    for ax, (range_label, _) in zip(axes, RANGES):
        for row in (r for r in table["rows"] if r["range"] == range_label):
            ls = WEDGE_STYLES[WEDGES.index(tuple(row["mu_wedge"]))]
            ax.plot(table["k_h_mpc"], row["response"], color=sigma_color(row["sigma0"]), ls=ls,
                    lw=1.1, label=rf"$\sigma_0={row['sigma0']:g}$" if ls == "-" else None)
        # Lighter and thinner than the dashed wedge, and behind the curves.
        ax.axvline(k_bao, color="0.6", lw=0.7, ls="--", zorder=0)
        ax.set(xscale="log", xlim=(0.01, 0.5), ylim=(0, 1.05), ylabel=r"$G(k)$",
               xticks=[0.01, 0.03, 0.1, 0.3])
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
        ax.xaxis.set_minor_formatter(plt.NullFormatter())
        ax.text(0.98, 0.95, rf"${range_label}$", transform=ax.transAxes, ha="right", va="top",
                fontsize=8)
    axes[0].text(k_bao, 1.01, r"$2\pi/r_d$", transform=axes[0].get_xaxis_transform(),
                 ha="center", va="bottom", fontsize=8, color="0.4")
    axes[0].legend(frameon=False, fontsize=7, loc="lower left")
    axes[1].legend(handles=[plt.Line2D([], [], color="0.3", ls=ls, lw=1.1,
                                       label=rf"${lower:.2g}\leq|\mu|<{upper:.2g}$")
                            for (lower, upper), ls in zip(WEDGES, WEDGE_STYLES)],
                   frameon=False, fontsize=7, loc="lower left")
    axes[1].set_xlabel(r"$k\ [h\,{\rm Mpc}^{-1}]$")
    show(fig, "fig_los_damping")


if BASELINE_REPORT is not None:
    if SELECTION_RUN is None:
        raise ValueError("The radial selection requires the matched frozen selection run")
    plt.style.use(PRD_STYLE_FILE)
    # Radial selection n(z) per tier; the attained density of each measurement bin at its z_eff.
    tiers = {row["label"]: row for row in frozen_config["selection"]["tiers"]}
    tracer_of = {shell["name"]: name for name, spec in frozen_config["tracers"].items()
                 for shell in spec["measurement_bins"]}
    targets = []  # one entry per bin and tier: attained density at z_eff, and its target
    for row in baseline["density"]:
        name, condition = row["measurement_bin"], row["condition"]
        z_eff = preparations[tracer_of[name]]["effective_redshifts"][condition][name]["z_eff"]
        target = tiers[condition]["target_nbar_h3_mpc3"]
        value = None if target is None else float(target[name])
        attained = row["attained_nbar_h3_mpc3"]
        targets.append(dict(
            measurement_bin=name, condition=condition, z_eff=float(z_eff),
            attained_nbar_h3_mpc3=attained, target_nbar_h3_mpc3=value,
            target_differs=value is not None and abs(value / attained - 1) > 1e-3,
        ))
    values = np.concatenate([row["density"] for row in baseline["density"]]
                            + [[t["target_nbar_h3_mpc3"] for t in targets
                                if t["target_nbar_h3_mpc3"] is not None]])
    ylim = (values.min() / 2, values.max() * 2)
    fig, radial = plt.subplots(1, 5, figsize=two_col(2.3), sharey=True,
                               gridspec_kw=dict(left=0.085, right=0.995, bottom=0.20, top=0.86,
                                                wspace=0.07))
    for ax, (condition, sigma) in zip(radial, zip(TIERS, SIGMA_LABEL_TO_VALUE.values())):
        rows = sorted((r for r in baseline["density"] if r["condition"] == condition),
                      key=lambda r: BINS.index(r["measurement_bin"]))
        for index, row in enumerate(rows):
            ax.stairs(row["density"], row["edges"], fill=True, lw=0, baseline=ylim[0],
                      color=sigma_color(sigma, "prerecon" if index % 2 == 0 else "postrecon"))
        for edge in (0.1, 0.2, 0.4, 0.6, 0.8):
            ax.axvline(edge, color="0.5", ls="--", lw=0.6, zorder=3)
        for t in (t for t in targets if t["condition"] == condition):
            ax.plot(t["z_eff"], t["attained_nbar_h3_mpc3"], "o", ms=3.2, color="black", zorder=6)
            if t["target_differs"]:
                ax.plot(t["z_eff"], t["target_nbar_h3_mpc3"], "o", ms=3.5, mec="black", mew=0.7,
                        mfc="white", zorder=5)
        ax.set(yscale="log", xlim=(0.1, 1.0), ylim=ylim, xticks=[0.2, 0.4, 0.6, 0.8], xlabel="$z$")
        title = r"$\sigma_0=0$ (original mock)" if condition == "native" else rf"$\sigma_0={sigma:g}$"
        ax.set_title(title, color=sigma_color(sigma), pad=3, fontsize=8)
        ax.grid(False)
    radial[0].set_ylabel(r"$\bar n(z)\ [h^3\,{\rm Mpc}^{-3}]$")
    radial[-1].legend(handles=[
        plt.Line2D([], [], ls="none", marker="o", color="black", ms=4,
                   label=r"Attained $\bar n$ at $z_{\rm eff}$"),
        plt.Line2D([], [], ls="none", marker="o", mfc="white", mec="black", ms=4,
                   label="Target, if not attained"),
    ], loc="lower left", frameon=False, fontsize=6.5, handlelength=1.0)
    supporting_tables["survey-window"] = dict(targets=targets)
    show(fig, "fig_nz", "Radial selection: attained n(z) per tier, with the attained density of "
                        "each measurement bin at its effective redshift.")

    import shutil
    import subprocess
    import tempfile

    # The analysis schematic is authored as an SVG; the export cell converts that source to the
    # vector PDF and the 300 dpi PNG with rsvg-convert. Here it is rasterized for display only.
    PIPELINE_SVG = REPO / "paper/figures/source/fig_pipeline.svg"
    fig, ax = plt.subplots(figsize=(5.4, 9.4))
    ax.axis("off")
    if PIPELINE_SVG.is_file() and shutil.which("rsvg-convert"):
        with tempfile.TemporaryDirectory() as tmp:
            raster = Path(tmp) / "fig_pipeline.png"
            subprocess.run(["rsvg-convert", "-f", "png", "-d", "150", "-p", "150", "-o", str(raster),
                            str(PIPELINE_SVG)], check=True)
            ax.imshow(plt.imread(raster), interpolation="none")
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
        caption = ("Analysis sequence per tracer, realization and redshift-error condition, from "
                   "paper/figures/source/fig_pipeline.svg.")
    else:
        from matplotlib.patches import Rectangle

        ax.set(xlim=(0, 1), ylim=(0, 1))
        ax.add_patch(Rectangle((0.02, 0.08), 0.96, 0.84, fill=False, edgecolor="0.5", lw=0.8))
        ax.text(0.5, 0.5, "Pipeline schematic: placeholder", ha="center", va="center", fontsize=10)
        caption = "Placeholder: the SVG source or rsvg-convert is unavailable."
    show(fig, "fig_pipeline", caption)

    from pzbao.theory import gaussian_damping_wedge_average, sigma_r_mpc_h

    cosmology = build_mock_cosmology(frozen_config)
    speed_of_light = frozen_config["physics"]["speed_of_light_km_s"]
    k = np.geomspace(0.01, 0.5, 300)  # logarithmic wavenumber axis, as in arXiv:2608.24276 Fig. 2
    response_rows = []
    for range_label, names in RANGES:
        for condition, sigma in zip(TIERS[1:], SIGMA_TIERS):
            weights, redshifts = [], []
            for name in names:
                prep = preparations[tracer_of[name]]
                redshifts.append(float(prep["effective_redshifts"][condition][name]["z_eff"]))
                weights.append(sum(prep["conditions"][condition][cap]["counts"][name]
                                   for cap in ("NGC", "SGC")))
            z = float(np.average(redshifts, weights=weights))
            sigma_r = float(sigma_r_mpc_h(z, sigma, float(cosmology.efunc(z)), speed_of_light))
            for lower, upper in WEDGES:
                response = gaussian_damping_wedge_average(k, sigma_r, lower, upper)
                response_rows.append(dict(range=range_label, bins=list(names), condition=condition,
                                          sigma0=sigma, z_eff=z, sigma_r_mpc_h=sigma_r,
                                          mu_wedge=[lower, upper], response=response))
    supporting_tables["los-damping"] = dict(
        k_h_mpc=k, rows=response_rows, drag_scale_mpc_h=float(cosmology.rs_drag),
        definition="G(k, wedge) = wedge average of exp[-(k mu sigma_r)^2 / 2], sigma_r = c sigma_0 "
                   "(1+z)/H(z) at the count-weighted effective redshift of the range's bins; the "
                   "BAO wavenumber is 2 pi / r_d with r_d = rs_drag of the same cosmology",
        interpretation="Analytic photo-z propagator of the galaxy field; the measured propagator "
                       "(specification revision photoz-propagator-20261015-v1) is pending",
    )
    los_damping_figure(supporting_tables["los-damping"], float(cosmology.rs_drag))
elif globals().get("saved_baseline_rows") is not None:
    # The committed rows predate the stored drag scale, so r_d comes from the configured cosmology.
    los_damping_figure(saved_baseline_rows["los-damping"], float(build_mock_cosmology().rs_drag))

In [ ]:
if BASELINE_REPORT is not None and selection is not None:
    # Cross-check the independently saved radial curves and preparation counts.
    for row in baseline["density"]:
        tracer = "LRG" if row["measurement_bin"].startswith("LRG") else "BGS_BRIGHT-21.5"
        caps = [s for s in selection if s["tracer"] == tracer and s["condition"] == row["condition"]]
        np.testing.assert_allclose(
            row["mean_count"], sum(s["counts"][row["measurement_bin"]] for s in caps), atol=1e-6
        )
        np.testing.assert_allclose(
            row["volume_mpc_h3"], sum(s["volumes_mpc_h3"][row["measurement_bin"]] for s in caps),
            rtol=1e-10,
        )



In [ ]:
if BASELINE_REPORT is not None:
    # Retain interval availability in the supporting data after removing the diagnostic figures.
    supporting_tables["interval-counts"] = readers.baseline_interval_counts(baseline["profile_diagnostics"])


In [ ]:
def literature_comparison_table(profiles, joint_rows):
    # Published quantities retain their own definitions and survey geometry.
    # eBOSS configuration-space BAO, Eqs. 52--53 of arXiv:2007.08993v2.
    distance = np.array([17.86, 19.34])
    covariance = np.array([[0.111, -0.0586], [-0.0586, 0.292]])
    gradient = np.array([2 / 3, 1 / 3]) / distance
    references = [
        dict(group="Spectroscopic", label=r"DESI DR1, $0.4<z<0.6$ ($\alpha_{\rm iso}$)",
             sigma0="spec.", precision=0.0113, marker="*",
             url="https://arxiv.org/html/2404.03000v1#S7.T15",
             definition="Table 15: LRG1 marginal alpha_iso SD, configuration space, post-reconstruction"),
        dict(group="Spectroscopic", label=r"eBOSS, $0.6<z<1.0$ ($D_V$)", sigma0="spec.",
             precision=float(np.sqrt(gradient @ covariance @ gradient)), marker="s",
             url="https://arxiv.org/html/2007.08993v2#S5.E52",
             definition="Linear propagation of Eqs. 52-53 to fractional D_V"),
        dict(group="Photometric", label=r"DES Y6 ($D_M$)", sigma0=r"$\approx$0.03",
             precision=0.021, marker="D",
             url="https://arxiv.org/abs/2402.10696",
             definition="Fractional transverse distance, 0.6<z_ph<1.2, ~3% photo-z precision"),
        dict(group="Photometric", label=r"DES Y3 ($D_M$)", sigma0=r"$\approx$0.03",
             precision=0.027, marker="d",
             url="https://arxiv.org/html/2107.04646v2", definition="Fractional transverse distance"),
        dict(group="Photometric", label="SDSS photometric", sigma0=r"$\approx$0.03",
             precision=0.065, marker="^",
             url="https://arxiv.org/html/astro-ph/0605302v2",
             definition="Fractional comoving distance; approximate redshift error"),
        dict(group="Photometric", label="Blake & Bridle (forecast)", sigma0="0.03",
             precision=0.020, marker="v",
             url="https://arxiv.org/pdf/astro-ph/0411713",
             definition="Acoustic-scale forecast: 10000 deg2, r<22.5, sigma0=0.03"),
    ]
    # Single-realization precision: the median Minos half-width over usable intervals,
    # comparable to an error bar quoted by one survey of the mock footprint. The 0.4 <= z < 0.6
    # bin is shown tier by tier as the peek at the tiers' performance; every other redshift
    # range contributes its joint row only, in redshift order, with a placeholder where the
    # designated report carries no per-realization profiles.
    from pzbao.fitting import nonnegative_blue_weights

    PEEK_BIN = "LRG1"
    # One-survey SPHEREx prediction (render_spherex_forecast.py): the matched-V_eff pipeline where
    # it reaches the SPHEREx effective volume, otherwise the conditional forecast; absent unless
    # the renderer supplies it. Predictions beyond OFFSCALE per cent carry no BAO constraint.
    forecast_path = globals().get("spherex_forecast_choice")
    OFFSCALE = 50.0
    forecast_rows = {}
    if forecast_path is not None:
        forecast_rows = {
            (r["measurement_bin"], r["condition"], r["reconstruction_state"]): r["prediction"]
            for r in json.loads(Path(forecast_path).read_text())["rows"]
            if r.get("prediction") is not None
        }

    def forecast_width(name, condition, state_name):
        prediction = forecast_rows.get((name, condition, state_name))
        if prediction is None:
            return None
        return dict(sigma=float(prediction["sigma_qiso"]), kind=prediction["kind"])

    def tier_rows_for(name):
        tracer, zmin, zmax, _ = ZBIN_GRID_PANELS[BINS.index(name)]
        rows = []
        for label, sigma in SIGMA_LABEL_TO_VALUE.items():
            if label == "desi_baseline":  # the unsmeared control is a reference, not a survey
                continue
            row = dict(condition=label, sigma0=sigma, measurement_bin=name)
            for state in ("prerecon", "postrecon"):
                key = (tracer, zmin, zmax, label, 15, state)
                if key not in profiles:
                    return None
                widths = np.array([m["qiso_minos"] for m in profiles[key].values()], dtype=float)
                usable = widths[np.isfinite(widths)]
                if usable.size == 0:
                    raise ValueError(f"No usable {name} interval for {label} {state}")
                row[state] = dict(median_minos_half_width=float(np.median(usable)),
                                  usable_intervals=int(usable.size), realizations=len(widths),
                                  spherex_prediction=forecast_width(
                                      name, label, STATES[("prerecon", "postrecon").index(state)]))
            rows.append(row)
        return rows

    def joint_forest_row(name, tier_rows):
        # Consensus combination of correlated single-survey error bars used by BOSS DR12 and
        # eBOSS DR16 (Sanchez et al. 2017, arXiv:1607.03146): the estimate covariance is the
        # tiers' median Minos half-widths times the measured cross-tier correlation of alpha_iso,
        # the weights are its non-negative best linear unbiased weights (specification revision
        # 2.14) and the joint half-width is the combination's own error.
        _, zmin, zmax, _ = ZBIN_GRID_PANELS[BINS.index(name)]
        joint = {r["route"]["reconstruction_state"]: r
                 for r in joint_rows if r["route"]["measurement_bin"] == name}
        row = dict(condition="joint", sigma0="0.001-0.03", measurement_bin=name, zmin=zmin,
                   zmax=zmax, status="available")
        if tier_rows is None or any(joint.get(s, {}).get("status") != "available" for s in STATES):
            row.update(status="unavailable",
                       reason="no per-realization profiles or joint record for this bin")
            return row
        for state, state_name in zip(("prerecon", "postrecon"), STATES):
            record = joint[state_name]
            tiers = record["tiers"]
            iso = [2 * t for t in range(len(tiers))]
            rho = np.array(record["estimate_correlation"], dtype=float)[np.ix_(iso, iso)]
            medians = np.array([
                next(r for r in tier_rows if r["condition"] == tier)[state]["median_minos_half_width"]
                for tier in tiers
            ])
            estimate_covariance = rho * np.outer(medians, medians)
            weights, removed = nonnegative_blue_weights(estimate_covariance)
            forecasts = [next(r for r in tier_rows if r["condition"] == tier)[state]
                         ["spherex_prediction"] for tier in tiers]
            kinds = {f["kind"] for f in forecasts if f is not None}
            forecast_joint = None if any(f is None for f in forecasts) else dict(
                sigma=readers.consensus_width([f["sigma"] for f in forecasts], rho),
                kind=kinds.pop() if len(kinds) == 1 else "mixed")
            row[state] = dict(
                spherex_prediction=forecast_joint,
                median_minos_half_width=float(np.sqrt(weights @ estimate_covariance @ weights)),
                usable_intervals=min(
                    next(r for r in tier_rows if r["condition"] == tier)[state]["usable_intervals"]
                    for tier in tiers
                ),
                realizations=25, weights=weights.tolist(),
                removed_tiers=[tiers[index] for index in removed],
                definition="consensus combination (Sanchez et al. 2017) of the tiers' median Minos "
                           "half-widths with the measured alpha_iso correlation; non-negative weights",
            )
        return row

    peek_rows = tier_rows_for(PEEK_BIN)
    if peek_rows is None:
        raise ValueError(f"No per-realization profiles for the peek bin {PEEK_BIN}")
    current = peek_rows + [joint_forest_row(PEEK_BIN, peek_rows)]
    joint_by_bin = [joint_forest_row(name, tier_rows_for(name)) for name in BINS]
    return dict(
        current=current, joint_by_bin=joint_by_bin, peek_bin=PEEK_BIN, references=references,
        current_definition="Median native Minos half-width of alpha_iso over usable intervals "
                           f"(valid, no prior-boundary contact) of the 25 individual {PEEK_BIN} fits",
        joint_definition="Consensus combination of the tiers' median Minos half-widths with the "
                         "measured alpha_iso correlation (Sanchez et al. 2017; BOSS DR12 and eBOSS "
                         "DR16 practice), non-negative weights; usable interval counts per tier are "
                         "kept in current",
        interpretation="Context only: distinct estimators, footprints, densities and redshifts",
        spherex_forecast=str(forecast_path) if forecast_path is not None else None,
        spherex_prediction_definition=(
            "one survey at the SPHEREx effective volume: the matched-V_eff pipeline's median "
            "usable Minos half-width where it attains at least 90 per cent of that volume, else "
            "the one-survey conditional density-and-area forecast (revisions 2.11-F1/F3); joint "
            "rows combine the tier predictions with the measured correlation; predictions above "
            f"{OFFSCALE:g} per cent are drawn at the axis edge"
        ),
    )


def literature_comparison_figure(table):
    """Draw the literature forest from its table (also the saved-rows replot path)."""
    current, joint_by_bin = table["current"], table["joint_by_bin"]
    references, PEEK_BIN = table["references"], table["peek_bin"]
    OFFSCALE = 50.0
    _, peek_zmin, peek_zmax, _ = ZBIN_GRID_PANELS[BINS.index(PEEK_BIN)]
    plt.style.use(PRD_STYLE_FILE)

    rows, y = [], 0.0
    rows.append(("header", ("This work: mock realizations", r"$\sigma_z/(1+z)$"), y))
    y += 1.0
    for joint_row in joint_by_bin:
        if joint_row["measurement_bin"] == PEEK_BIN:
            rows.append(("subheader", rf"${peek_zmin:.1f}\leq z<{peek_zmax:.1f}$", y))
            y += 0.9
            for item in current:
                rows.append(("row", item, y))
                y += 1.0
        else:
            rows.append(("row", joint_row, y))
            y += 1.0
    y += 0.35
    for header, items in (
        ("Spectroscopic", [r for r in references if r["group"] == "Spectroscopic"]),
        ("Photometric", [r for r in references if r["group"] == "Photometric"]),
    ):
        rows.append(("header", (header, ""), y))
        y += 1.0
        for item in items:
            rows.append(("row", item, y))
            y += 1.0
        y += 0.35
    # Diamonds: the one-survey SPHEREx prediction, light before and dark after reconstruction;
    # filled from the matched-V_eff pipeline, open from the theoretical estimate and half-filled
    # where a combination of conditions mixes the two. Only drawn kinds enter the legend.
    DIAMONDS = {
        "matched_veff_pipeline": (r"SPHEREx, matched $V_{\rm eff}$", {}),
        "mixed": (r"SPHEREx, matched $V_{\rm eff}$ and forecast",
                  dict(fillstyle="left", markerfacecoloralt="white")),
        "conditional_forecast": ("SPHEREx, forecast", dict(mfc="white")),
    }
    drawn_kinds = set()

    def plot_forecast(item, position, light, dark):
        for state, color, order in (("prerecon", light, 2), ("postrecon", dark, 3)):
            prediction = item[state].get("spherex_prediction")
            if prediction is None:
                continue
            value = 100 * prediction["sigma"]
            if value > OFFSCALE:
                ax.plot(OFFSCALE, position, "<", ms=4, color=color, mec="black", mew=0.6,
                        zorder=order)
                continue
            drawn_kinds.add(prediction["kind"])
            ax.plot(value, position, "D", ms=4, color=color, mec="black", mew=0.6, zorder=order,
                    **DIAMONDS[prediction["kind"]][1])

    def joint_name(item):
        return rf"${item['zmin']:.1f}\leq z<{item['zmax']:.1f}$, joint"

    fig, ax = plt.subplots(figsize=style.one_col(4.9))
    fig.subplots_adjust(left=0.40, right=0.80, top=0.985, bottom=0.095)
    for kind, item, position in rows:
        if kind == "header":
            ax.text(-0.04, position, item[0], transform=ax.get_yaxis_transform(), ha="right",
                    va="center", fontsize=7.5, fontweight="bold")
            ax.text(1.04, position, item[1], transform=ax.get_yaxis_transform(), ha="left",
                    va="center", fontsize=6.5, color="0.35")
            continue
        if kind == "subheader":
            ax.text(-0.04, position, item, transform=ax.get_yaxis_transform(), ha="right",
                    va="center", fontsize=7.5, style="italic")
            continue
        if "group" in item:
            ax.plot(100 * item["precision"], position, item["marker"], color="0.72",
                    mec="black", mew=0.6, ms=6 if item["marker"] != "*" else 8)
            name, sigma_text = item["label"], item["sigma0"]
        elif item["condition"] == "joint" and item["measurement_bin"] != PEEK_BIN:
            name = joint_name(item)
            sigma_text = item["sigma0"]
            if item["status"] != "available":
                ax.text(0.5, position, "pending", transform=ax.get_yaxis_transform(),
                        ha="center", va="center", fontsize=6.5, color="0.5")
            else:
                pre, post = (100 * item[s]["median_minos_half_width"] for s in ("prerecon", "postrecon"))
                ax.plot([pre, post], [position, position], color="0.6", lw=0.8, zorder=1)
                ax.plot(pre, position, "o", ms=5, color="#c4c4c4", mec="black", mew=0.6, zorder=2)
                ax.plot(post, position, "o", ms=5, color=JOINT_COLOR, mec="black", mew=0.6, zorder=3)
                plot_forecast(item, position, "#c4c4c4", JOINT_COLOR)
        else:
            sigma = item["sigma0"]
            pre, post = (100 * item[s]["median_minos_half_width"] for s in ("prerecon", "postrecon"))
            ax.plot([pre, post], [position, position], color="0.6", lw=0.8, zorder=1)
            if item["condition"] == "joint":
                light, dark = "#c4c4c4", JOINT_COLOR
                name, sigma_text = joint_name(item), sigma
            else:
                light, dark = sigma_color(sigma, "prerecon"), sigma_color(sigma, "postrecon")
                name, sigma_text = rf"$\sigma_0={sigma:g}$", f"{sigma:g}"
            # Filled markers with black rims: the light member of the pair is pre-reconstruction
            # and the dark member post-reconstruction.
            ax.plot(pre, position, "o", ms=5, color=light, mec="black", mew=0.6, zorder=2)
            ax.plot(post, position, "o", ms=5, color=dark, mec="black", mew=0.6, zorder=3)
            plot_forecast(item, position, light, dark)
        ax.text(-0.04, position, name, transform=ax.get_yaxis_transform(), ha="right",
                va="center", fontsize=7.5)
        ax.text(1.04, position, sigma_text, transform=ax.get_yaxis_transform(), ha="left",
                va="center", fontsize=7)
    shown = current + [r for r in joint_by_bin
                       if r["status"] == "available" and r["measurement_bin"] != PEEK_BIN]
    values = [100 * r["precision"] for r in references] + [
        100 * row[s]["median_minos_half_width"] for row in shown for s in ("prerecon", "postrecon")] + [
        min(100 * row[s]["spherex_prediction"]["sigma"], OFFSCALE) for row in shown
        for s in ("prerecon", "postrecon") if row[s].get("spherex_prediction") is not None]
    # Reversed axis: smaller fractional errors (better precision) lie to the right.
    ax.set(xscale="log", xlim=(max(values) * 1.5, min(values) / 1.5), ylim=(y - 0.2, -1.0),
           yticks=[], xlabel="Fractional distance-scale error [%]")
    ax.annotate("", xy=(0.97, -0.118), xytext=(0.03, -0.118), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="->", color="0.35", lw=0.9), annotation_clip=False)
    ax.text(0.03, -0.128, "worse", transform=ax.transAxes, ha="left", va="top", fontsize=7,
            color="0.35")
    ax.text(0.97, -0.128, "better", transform=ax.transAxes, ha="right", va="top", fontsize=7,
            color="0.35")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.grid(True, axis="x", which="major", alpha=0.3)
    ax.legend(handles=[
        plt.Line2D([], [], ls="none", marker="o", color="#c4c4c4", mec="black", mew=0.6, ms=5,
                   label="Pre-recon. (light)"),
        plt.Line2D([], [], ls="none", marker="o", color=JOINT_COLOR, mec="black", mew=0.6, ms=5,
                   label="Post-recon. (dark)"),
        *[plt.Line2D([], [], ls="none", marker="D", color="0.6", mec="black", mew=0.6, ms=4,
                     label=label, **fill)
          for kind, (label, fill) in DIAMONDS.items() if kind in drawn_kinds],
    ], loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, fontsize=7,
              handletextpad=0.2)
    show(fig, "fig_literature_comparison")


def literature_comparison(profiles, joint_rows):
    table = literature_comparison_table(profiles, joint_rows)
    supporting_tables["literature-comparison"] = table
    literature_comparison_figure(table)


if BASELINE_REPORT is not None:
    literature_comparison(profiles_full, joint_rows)
elif globals().get("saved_baseline_rows") is not None:
    literature_comparison_figure(saved_baseline_rows["literature-comparison"])

In [ ]:
def alpha_to_distance_ratios(mean, covariance):
    """Distance ratios to the mock cosmology in the DESI convention, with propagated errors.

    desilike fits qap = qpar/qper: alpha_iso = D_V/r_d, alpha_AP = D_H/D_M,
    alpha_par = alpha_iso alpha_AP^(2/3) = D_H/r_d and alpha_perp = alpha_iso alpha_AP^(-1/3)
    = D_M/r_d, each relative to its fiducial value, so D_M/D_H = 1/alpha_AP. The errors follow
    from the (alpha_iso, alpha_AP) covariance to first order.
    """
    iso, ap = (float(v) for v in mean)
    cov = np.asarray(covariance, dtype=float)
    values = {
        "D_V/r_d": iso,
        "D_M/D_H": 1 / ap,
        "D_M/r_d": iso * ap ** (-1 / 3),
        "D_H/r_d": iso * ap ** (2 / 3),
    }
    gradients = {
        "D_V/r_d": (1.0, 0.0),
        "D_M/D_H": (0.0, -1 / ap**2),
        "D_M/r_d": (ap ** (-1 / 3), -iso * ap ** (-4 / 3) / 3),
        "D_H/r_d": (ap ** (2 / 3), 2 * iso * ap ** (-1 / 3) / 3),
    }
    return {
        key: dict(value=value,
                  error=float(np.sqrt(np.array(gradients[key]) @ cov @ np.array(gradients[key]))))
        for key, value in values.items()
    }


def distance_ratio_figure(joint_rows, z_eff_rows):
    """Four distance ratios to the mock cosmology against z_eff, points only (DESI DR2 Fig. 6 layout)."""
    rows, unavailable = [], []
    for record in joint_rows:
        route = record["route"]
        if record["status"] != "available":
            unavailable.append(dict(route, reason=record["reason"]))
            continue
        state = route["reconstruction_state"]
        z = [
            r["z_eff"] for r in z_eff_rows
            if r["measurement_bin"] == route["measurement_bin"] and r["condition"] in record["tiers"]
        ]
        if len(z) != len(record["tiers"]):
            raise ValueError("effective redshifts incomplete for a joint record")
        rows.append(dict(route, z_eff=float(np.mean(z)), z_eff_spread=float(np.ptp(z)),
                         ratios=alpha_to_distance_ratios(record["mean"], record["covariance"])))
    supporting_tables["distance-ratios"] = dict(
        rows=rows,
        unavailable=unavailable,
        convention="alpha_iso = D_V/r_d, alpha_AP = D_H/D_M (desilike qap = qpar/qper), alpha_perp = "
                   "alpha_iso alpha_AP^(-1/3) = D_M/r_d, alpha_par = alpha_iso alpha_AP^(2/3) = D_H/r_d "
                   "and D_M/D_H = 1/alpha_AP, each relative to "
                   "the mock (Planck 2018) cosmology; errors by first-order propagation of the "
                   "joint (alpha_iso, alpha_AP) covariance; z_eff is the mean of the four tiers' "
                   "effective redshifts",
        source="joint combination of the four photometric tiers (specification revision 2.14)",
    )
    draw_distance_ratios(rows)


def draw_distance_ratios(rows):
    """Draw the four distance-ratio panels from the joint rows (also the saved-rows replot path)."""
    panels = (
        ("D_V/r_d", r"$(D_V/r_d)\,/\,(D_V/r_d)^{\rm fid}$"),
        ("D_M/D_H", r"$(D_M/D_H)\,/\,(D_M/D_H)^{\rm fid}$"),
        ("D_M/r_d", r"$(D_M/r_d)\,/\,(D_M/r_d)^{\rm fid}$"),
        ("D_H/r_d", r"$(D_H/r_d)\,/\,(D_H/r_d)^{\rm fid}$"),
    )
    fig, axes = plt.subplots(2, 2, figsize=two_col(5.2), sharex=True, layout="constrained")
    offset = {"prereconstruction": -0.006, "postreconstruction": 0.006}
    # One DESI tracer hue per redshift bin. Pre-reconstruction points are open, with error bars
    # in the light member of the pair; post-reconstruction points are filled in the dark member.
    for ax, (key, label) in zip(axes.flat, panels):
        ax.axhline(1.0, color="0.5", lw=0.8)
        for row in rows:
            post = row["reconstruction_state"] == "postreconstruction"
            light, dark = style.zbin_pair(BINS.index(row["measurement_bin"]))
            ratio = row["ratios"][key]
            ax.errorbar(
                row["z_eff"] + offset[row["reconstruction_state"]], ratio["value"],
                yerr=ratio["error"], fmt="o", ms=4.5, mfc=dark if post else "white",
                mec="black" if post else dark, mew=0.6 if post else 1.0,
                ecolor=dark if post else light, capsize=2, lw=0.9, zorder=3 if post else 2,
            )
        # Symmetric about the fiducial value, so that unity sits at the centre of every panel.
        half = 1.08 * max(abs(r["ratios"][key]["value"] - 1) + r["ratios"][key]["error"] for r in rows)
        ax.set_ylim(1 - half, 1 + half)
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.3)
    axes[0, 0].set_xlim(0, 1.0)
    for ax in axes[1]:
        ax.set_xlabel(r"$z_{\rm eff}$")
    shown_bins = sorted({row["measurement_bin"] for row in rows}, key=BINS.index)
    handles = [
        plt.Line2D([], [], ls="none", marker="o", mfc=style.zbin_pair(BINS.index(name))[1],
                   mec="black", mew=0.6, ms=4.5, label=ZLABELS[BINS.index(name)])
        for name in shown_bins
    ] + [
        plt.Line2D([], [], ls="none", marker="o", mfc="white", mec=JOINT_COLOR, mew=1.0, ms=4.5,
                   label="Pre-reconstruction"),
        plt.Line2D([], [], ls="none", marker="o", mfc=JOINT_COLOR, mec="black", mew=0.6, ms=4.5,
                   label="Post-reconstruction"),
    ]
    fig.legend(handles=handles, loc="outside upper center", frameon=False, fontsize=6.5,
               ncol=len(handles), handletextpad=0.2, columnspacing=1.0)
    show(
        fig,
        "fig_distance_ratios",
        "Joint-tier distance ratios to the mock (Planck 2018) cosmology against effective "
        "redshift, in the DESI DR2 layout: D_V/r_d, D_M/D_H, D_M/r_d and D_H/r_d from the joint "
        "(alpha_iso, alpha_AP) constraint of each bin, pre- (hollow) and post-reconstruction "
        "(filled), one colour per redshift range; no model curves. Expected ratio unity for the "
        "exact mock cosmology. Bins whose "
        "designated report carries no per-realization profiles have no joint point.",
    )


if BASELINE_REPORT is not None and SELECTION_RUN is not None:
    distance_ratio_figure(joint_rows, z_eff_rows)
elif globals().get("saved_baseline_rows") is not None:
    draw_distance_ratios(saved_baseline_rows["distance-ratios"]["rows"])


In [ ]:
if BASELINE_REPORT is not None:
    # Reuse the maintained forecast notebook's drawing cell; input validation stays in data.py.
    forecast_notebook = json.loads((REPO / "pzbao/figures/forecasts.ipynb").read_text())
    for cell in forecast_notebook["cells"]:
        if cell["id"] in ("precision_caption", "precision"):
            exec(compile("".join(cell["source"]), "forecasts:" + cell["id"], "exec"), globals())
    precision_plot_precision(precision_rows, name="fig_sigma_r_precision")


In [ ]:
def posterior_constraints_tex(records, summary):
    """Generate each bin's table from checked native posterior moments."""
    fields = ("measurement_bin", "condition", "reconstruction_state")
    means = {tuple(row[key] for key in fields) + (row["parameter"],): row["mean"]
             for row in summary}
    expected = {tuple(row["route"][key] for key in fields) + (parameter,)
                for row in records for parameter in PARAMETERS}
    if len(means) != len(summary) or set(means) != expected:
        raise ValueError("Incomplete or duplicate posterior mean summary")
    lines = []
    for name in BINS:
        lines += [
            r"\begin{table*}[t]",
            r"\caption{\label{tab:baseline_" + name + "}" + name +
            r" mean-data posterior constraints. Errors are marginal standard deviations. "
            r"$\dagger$ denotes a flagged or unavailable mean-profile interval; reliable ESS is unavailable for every row.}",
            r"\begin{ruledtabular}",
            r"\begin{tabular}{llcccc}",
            r"$\sigma_0$ & State & $\aiso$ & $\aap$ & $\rho$ & Joint FoM \\",
            r"\hline",
        ]
        for tier, state in itertools.product(TIERS, STATES):
            row = next(r for r in records if
                       (r["route"]["measurement_bin"], r["route"]["condition"], r["route"]["reconstruction_state"])
                       == (name, tier, state))
            covariance = np.asarray(row["covariance"])
            errors = np.sqrt(np.diag(covariance))
            rho = covariance[0, 1] / np.prod(errors)
            label = "0" if tier == "native" else tier.split("_")[1]
            flag = r"$^{\dagger}$" if row["profile_flagged"] else ""
            values = [r"$%.4f\pm%.4f$" % (means[(name, tier, state, parameter)], sd)
                      for parameter, sd in zip(PARAMETERS, errors)]
            lines.append(" & ".join([label, ("Pre" if state == STATES[0] else "Post") + flag,
                                    *values, f"{rho:+.3f}", f"{row['fom']:.1f}"]) + r" \\")
        lines += [r"\end{tabular}", r"\end{ruledtabular}", r"\end{table*}"]
    return "\n".join(lines) + "\n"


def density_constraints_tex(rows):
    lines = [
        r"\begin{table*}[t]",
        r"\caption{\label{tab:baseline_density}Attained mean density per redshift bin and "
        r"redshift-error tier, averaged over each measurement volume and both caps. "
        r"The $\sigma_0=0$ column is the original mock, with no added redshift error.}",
        r"\begin{ruledtabular}", r"\begin{tabular}{lccccc}",
        r"Redshift bin & \multicolumn{5}{c}{$\bar n\ [10^{-4}\,h^3\,{\rm Mpc}^{-3}]$} \\",
        r" & $\sigma_z/(1+z)=0$ & $0.001$ & $0.003$ & $0.01$ & $0.03$ \\", r"\hline",
    ]
    for name, zlabel in zip(BINS, ZLABELS):
        values = [next(r["attained_nbar_h3_mpc3"] for r in rows
                       if (r["measurement_bin"], r["condition"]) == (name, tier)) for tier in TIERS]
        lines.append(zlabel + " & " + " & ".join(f"{v*1e4:.3f}" for v in values) + r" \\")
    return "\n".join(lines + [r"\end{tabular}", r"\end{ruledtabular}", r"\end{table*}", ""])


table_tex = {}
if BASELINE_REPORT is not None:
    table_tex["posterior-constraints"] = posterior_constraints_tex(
        baseline["posterior_records"], baseline["posterior_summary"])
    table_tex["density-constraints"] = density_constraints_tex(baseline["density"])


## Save figures

Choose an output directory under `products/` to save PNG and PDF copies. Rerunning this cell updates those figures.


In [ ]:
# Choose a destination explicitly. Interactive re-execution updates this draft bundle.
OUTPUT_DIR = None  # REPO / "paper" / "figures"
OUTPUT_DIR = globals().get("figure_output_choice", OUTPUT_DIR)
if OUTPUT_DIR is not None:
    from datetime import datetime, timezone
    import subprocess

    destination = Path(OUTPUT_DIR)
    destination.mkdir(parents=True, exist_ok=True)
    if globals().get("BASELINE_REPORT") is not None:
        baseline["sources"].unchanged()
    import shutil

    PIPELINE_SVG = REPO / "paper/figures/source/fig_pipeline.svg"
    for name, fig in figures.items():
        if name == "fig_pipeline" and PIPELINE_SVG.is_file() and shutil.which("rsvg-convert"):
            # Vector PDF and 300 dpi PNG straight from the authored SVG source.
            for suffix, options in (("pdf", []), ("png", ["-d", "300", "-p", "300"])):
                subprocess.run(["rsvg-convert", "-f", suffix, *options, "-o",
                                str(destination / f"{name}.{suffix}"), str(PIPELINE_SVG)], check=True)
            continue
        padding = dict(pad_inches=0.15) if name == "fig_pipeline" else {}
        fig.savefig(destination / f"{name}.png", dpi=300, bbox_inches="tight", **padding)
        fig.savefig(destination / f"{name}.pdf", bbox_inches="tight", **padding)
    if globals().get("BASELINE_REPORT") is not None:
        source_dir = destination / "source" / "baseline"
        source_dir.mkdir(parents=True, exist_ok=True)
        for name, value in supporting_tables.items():
            readers.write_json(source_dir / f"{name}.json", value)
        for name, value in table_tex.items():
            (source_dir / f"{name}.tex").write_text(value)
        consumed = baseline["sources"]
        consumed.unchanged()
        readers.write_json(source_dir / "provenance.json", dict(
            observed_utc=datetime.now(timezone.utc).isoformat(),
            baseline_report=str(Path(BASELINE_REPORT).resolve()),
            selection_run=str(SELECTION_RUN) if SELECTION_RUN is not None else None,
            measurement_identity=baseline["measurement_identity"],
            renderer_checkout=str(REPO),
            renderer_commit=subprocess.check_output(
                ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
            renderer_diff=subprocess.check_output(
                ["git", "diff", "--", "pzbao/figures", "paper/main.proposed.tex"],
                cwd=REPO, text=True),
            source_bytes=consumed.digests,
            scientific_acceptance="Not established by this rendering",
        ))
        readers.write_json(source_dir / "figure-index.json", {
            name: dict(
                caption=figure_captions.get(name, ""),
                axes=[dict(xlim=list(ax.get_xlim()), ylim=list(ax.get_ylim()),
                           xlabel=ax.get_xlabel(), ylabel=ax.get_ylabel())
                      for ax in figure.axes],
                pdf=name + ".pdf", png=name + ".png",
            ) for name, figure in figures.items()
        })
